In [1]:
# ============================================================
# OIP v1.0.37 — C01
# SOURCE MANIFEST & INTEGRITY LOCK
# Nigeria GHS-Panel Wave 5
# ============================================================

from pathlib import Path
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C01")
print("SOURCE MANIFEST & INTEGRITY LOCK")
print("Nigeria GHS-Panel Wave 5")
print("=" * 72)

# ------------------------------------------------------------
# 1. Runtime
# ------------------------------------------------------------

print("\n[1] RUNTIME")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("UTC:", datetime.now(timezone.utc).isoformat())

# ------------------------------------------------------------
# 2. Mounted dataset roots
# ------------------------------------------------------------

dataset_roots = sorted(
    [p for p in INPUT_ROOT.iterdir() if p.is_dir()]
)

if not dataset_roots:
    raise RuntimeError(
        "FAIL-CLOSED: No mounted dataset detected."
    )

print("\n[2] MOUNTED DATASET ROOTS")

for root in dataset_roots:
    print(" -", root)

# ------------------------------------------------------------
# 3. Mounted source inventory
# ------------------------------------------------------------

files = sorted(
    [p for p in INPUT_ROOT.rglob("*") if p.is_file()]
)

if not files:
    raise RuntimeError(
        "FAIL-CLOSED: No source files detected."
    )

print("\n[3] MOUNTED SOURCE INVENTORY")
print("Total mounted files:", len(files))

# File-type counts
pdf_files = sorted(
    [p for p in files if p.suffix.lower() == ".pdf"]
)

dta_files = sorted(
    [p for p in files if p.suffix.lower() == ".dta"]
)

zip_files = sorted(
    [p for p in files if p.suffix.lower() == ".zip"]
)

print("PDF files:", len(pdf_files))
print("DTA files:", len(dta_files))
print("ZIP files:", len(zip_files))

# ------------------------------------------------------------
# 4. SHA-256
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("\n[4] SHA-256 MANIFEST")

manifest = []

for f in files:

    digest = sha256_file(f)

    record = {
        "relative_path": str(f.relative_to(INPUT_ROOT)),
        "size_bytes": f.stat().st_size,
        "sha256": digest
    }

    manifest.append(record)

    print(
        digest,
        " ",
        f.relative_to(INPUT_ROOT)
    )

# ------------------------------------------------------------
# 5. Expected documentation verification
# ------------------------------------------------------------

expected_pdfs = {
    "ddi":
        "ddi-documentation-english_microdata-6410.pdf",

    "basic_information_v2":
        "ghs_panel_wave_5_basic_information_document_v2.pdf",

    "pp_household":
        "ghs_panel_w5_pp_household_questionnaire.pdf",

    "pp_agriculture":
        "ghs_panel_w5_pp_agriculture_questionnaire.pdf",

    "pp_community":
        "ghs_panel_w5_pp_community_questionnaire.pdf",

    "pp_supervisor":
        "ghs_panel_w5_pp_supervisor_manual.pdf",

    "pp_interviewer":
        "ghs_panel_w5_pp_interviewer_manual.pdf",

    "ph_household":
        "ghs_panel_w5_ph_household_questionnaire.pdf",

    "ph_agriculture":
        "ghs_panel_w5_ph_agriculture_questionnaire.pdf",

    "ph_community":
        "ghs_panel_w5_ph_community_questionnaire.pdf",

    "ph_supervisor":
        "ghs_panel_w5_ph_supervisor_manual.pdf",

    "ph_interviewer":
        "ghs_panel_w5_ph_interviewer_manual.pdf",

    "ph_measurer":
        "ghs_panel_w5_ph_measurer_manual.pdf"
}

filenames = {
    Path(x["relative_path"]).name
    for x in manifest
}

print("\n[5] EXPECTED DOCUMENTATION CHECK")

documentation_check = {}

for key, filename in expected_pdfs.items():

    present = filename in filenames

    documentation_check[key] = {
        "filename": filename,
        "present": present
    }

    print(
        f"{key}:",
        "PRESENT" if present else "MISSING"
    )

# ------------------------------------------------------------
# 6. Dataset structure state
# ------------------------------------------------------------

print("\n[6] DATASET MOUNT STATE")

if zip_files:
    mount_state = "ARCHIVE_PRESENT"
elif dta_files:
    mount_state = "ARCHIVE_NOT_MOUNTED_DATA_FILES_PRESENT"
else:
    mount_state = "NO_DATA_FILES_DETECTED"

print("Mount state:", mount_state)

# Important:
# Kaggle may mount the contents of an uploaded ZIP directly.
# Therefore ZIP absence is NOT treated as a failure when .dta files exist.

# ------------------------------------------------------------
# 7. Provenance flags
# ------------------------------------------------------------

ddi_present = documentation_check["ddi"]["present"]
bid_v2_present = documentation_check["basic_information_v2"]["present"]

all_docs_present = all(
    item["present"]
    for item in documentation_check.values()
)

print("\n[7] PROVENANCE FLAGS")

print("DDI_PRESENT:", ddi_present)
print("BID_V2_PRESENT:", bid_v2_present)
print("ALL_EXPECTED_DOCUMENTATION_PRESENT:", all_docs_present)
print("DTA_FILES_PRESENT:", len(dta_files) > 0)
print("ZIP_PRESENT_IN_MOUNT:", len(zip_files) > 0)

# ------------------------------------------------------------
# 8. Save manifest
# ------------------------------------------------------------

manifest_path = OUTPUT_ROOT / "C01_source_manifest_sha256.json"

c01_record = {

    "protocol":
        "OIP v1.0.37",

    "cell":
        "C01",

    "purpose":
        "Source inventory and mounted-source SHA-256 integrity manifest",

    "dataset":
        "Nigeria General Household Survey, Panel 2023-2024, Wave 5",

    "catalog_reference_claimed":
        "NGA_2023_GHSP-W5_v02_M",

    "kaggle_input_root":
        str(INPUT_ROOT),

    "dataset_roots":
        [str(x) for x in dataset_roots],

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "total_mounted_files":
        len(files),

    "pdf_count":
        len(pdf_files),

    "dta_count":
        len(dta_files),

    "zip_count":
        len(zip_files),

    "mount_state":
        mount_state,

    "documentation_check":
        documentation_check,

    "archive_sha256_available_in_mount":
        len(zip_files) > 0,

    "files":
        manifest,

    "actions_not_performed": [

        "ZIP extraction",
        "Variable parsing",
        "Construct authorization",
        "Outcome authorization",
        "Analytical cohort authorization",
        "Score calculation",
        "Empirical evaluation"
    ]
}

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as out:

    json.dump(
        c01_record,
        out,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("C01 FINAL STATUS")
print("=" * 72)

if not dta_files:
    print("STATUS: FAIL-CLOSED")
    print("Reason: No .dta data files detected.")

elif not all_docs_present:
    print("STATUS: FAIL-CLOSED")
    print("Reason: Required documentation is missing.")

else:
    print("STATUS: PASS")
    print("Mounted data files detected.")
    print("All expected documentation files detected.")
    print("ZIP absence is recorded as a mount-state condition,")
    print("not treated as a source failure.")

print("\nTotal mounted files:", len(files))
print("PDF files:", len(pdf_files))
print("DTA files:", len(dta_files))
print("ZIP files visible:", len(zip_files))
print("Mount state:", mount_state)

print("\nManifest:")
print(manifest_path)

print("\nC01 BOUNDARY:")
print("No variable analysis.")
print("No construct authorization.")
print("No outcome authorization.")
print("No cohort authorization.")
print("No score calculation.")
print("No empirical evaluation.")
print("No OIP validity claim.")

print("\nPRINCIPLE:")
print("No evidence → no operationalization.")

OIP v1.0.37 — C01
SOURCE MANIFEST & INTEGRITY LOCK
Nigeria GHS-Panel Wave 5

[1] RUNTIME
Python: 3.12.13
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
UTC: 2026-09-30T04:36:48.677960+00:00

[2] MOUNTED DATASET ROOTS
 - /kaggle/input/datasets

[3] MOUNTED SOURCE INVENTORY
Total mounted files: 121
PDF files: 13
DTA files: 108
ZIP files: 0

[4] SHA-256 MANIFEST
ff02df590a4928612f527d9842da7dc54391142457694e88943781e8c9c7764f   datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata/Post Harvest Wave 5/Agriculture/secta11c2_harvestw5.dta
61273fe2a29b52605126156dff0729e889bda43c3f39cb04468e67541961e220   datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata/Post Harvest Wave 5/Agriculture/secta11c3_harvestw5.dta
16605e9f10333d555a2a6a5bb2cb80f31d36f5c92d9c8cd38bce1b07f6af53c1   datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata/Post Harvest Wav

In [2]:
!pip install -q pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 31.8 MB/s eta 0:00:00


In [3]:
# ============================================================
# OIP v1.0.37 — C02
# Dataset & Documentation Structure Inventory
# Nigeria GHS-Panel Wave 5
# ============================================================
# Purpose:  Structural inventory of all mounted files.
# Method:   pyreadstat metadataonly — no data loading.
# Boundary: No variable / construct / outcome authorization.
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C02")
print("DATASET & DOCUMENTATION STRUCTURE INVENTORY")
print("=" * 72)

# ------------------------------------------------------------
# [1] Discover files
# ------------------------------------------------------------
files = sorted([p for p in INPUT_ROOT.rglob("*") if p.is_file()])
dta_files = sorted([p for p in files if p.suffix.lower() == ".dta"])
pdf_files = sorted([p for p in files if p.suffix.lower() == ".pdf"])
zip_files = sorted([p for p in files if p.suffix.lower() == ".zip"])

print("\n[1] SOURCE COUNTS")
print("Total files:", len(files))
print("DTA files:", len(dta_files))
print("PDF files:", len(pdf_files))
print("ZIP files visible:", len(zip_files))

if not dta_files:
    raise RuntimeError("FAIL-CLOSED: No .dta files detected.")

if not pdf_files:
    raise RuntimeError("FAIL-CLOSED: No documentation PDFs detected.")

# ------------------------------------------------------------
# [2] Directory structure inventory
# ------------------------------------------------------------
print("\n[2] DIRECTORY STRUCTURE")

dir_counts = {}

for p in dta_files:
    rel_parts = p.relative_to(INPUT_ROOT).parts
    d = str(Path(*rel_parts[:-1]))
    dir_counts[d] = dir_counts.get(d, 0) + 1

for d, n in sorted(dir_counts.items()):
    print(f"  {n:>3} dta | {d}")

# ------------------------------------------------------------
# [3] Stata structure inventory
#     metadataonly=True — NO DATA LOAD
# ------------------------------------------------------------
print("\n[3] STATA STRUCTURE INVENTORY")

try:
    import pyreadstat
except ImportError:
    raise RuntimeError("FAIL-CLOSED: pyreadstat is required.")

print("pyreadstat version:", getattr(pyreadstat, "__version__", "UNKNOWN"))

stata_inventory = []

for i, path in enumerate(dta_files, 1):
    rel = str(path.relative_to(INPUT_ROOT))

    try:
        df, meta = pyreadstat.read_dta(
            str(path),
            metadataonly=True
        )

        record = {
            "file": rel,
            "status": "READ_OK",
            "rows": int(meta.number_rows),
            "columns": int(meta.number_columns),
            "column_names_count": len(meta.column_names),
            "size_bytes": path.stat().st_size
        }

        print(
            f"[{i:03d}/{len(dta_files)}] OK | "
            f"rows={meta.number_rows:>8} "
            f"cols={meta.number_columns:>4} | "
            f"{Path(rel).name}"
        )

    except Exception as e:

        record = {
            "file": rel,
            "status": "READ_ERROR",
            "error": repr(e),
            "size_bytes": path.stat().st_size
        }

        print(
            f"[{i:03d}/{len(dta_files)}] ERROR | "
            f"{Path(rel).name}"
        )
        print("       ", repr(e))

    stata_inventory.append(record)

# ------------------------------------------------------------
# [4] Documentation inventory
# ------------------------------------------------------------
print("\n[4] DOCUMENTATION INVENTORY")

documentation_inventory = []

for path in pdf_files:

    rel = str(path.relative_to(INPUT_ROOT))

    rec = {
        "file": rel,
        "size_bytes": path.stat().st_size
    }

    documentation_inventory.append(rec)

    print(
        f"{path.stat().st_size:>12} bytes | "
        f"{Path(rel).name}"
    )

# ------------------------------------------------------------
# [5] Structural summary
# ------------------------------------------------------------
read_ok = sum(
    1 for x in stata_inventory
    if x["status"] == "READ_OK"
)

read_error = sum(
    1 for x in stata_inventory
    if x["status"] == "READ_ERROR"
)

total_rows = sum(
    x.get("rows", 0)
    for x in stata_inventory
    if x["status"] == "READ_OK"
)

total_cols = sum(
    x.get("columns", 0)
    for x in stata_inventory
    if x["status"] == "READ_OK"
)

total_size = sum(
    x.get("size_bytes", 0)
    for x in stata_inventory
)

print("\n[5] STRUCTURAL SUMMARY")
print("DTA files:", len(dta_files))
print("DTA readable:", read_ok)
print("DTA read errors:", read_error)
print(
    "Total observed rows (sum across files):",
    total_rows
)
print(
    "Total observed columns (sum across files):",
    total_cols
)
print(
    "Total .dta size on disk:",
    total_size,
    "bytes"
)
print("PDF files:", len(pdf_files))

# ------------------------------------------------------------
# [6] Dataset identity / version flags
# ------------------------------------------------------------
print("\n[6] DATASET IDENTITY FLAGS")

mount_paths = sorted({
    str(p.parts[2])
    for p in files
    if len(p.parts) > 2
})

print("Mount top-level entries:", mount_paths)

directory_v02_present = any(
    "v02_M" in str(p)
    for p in dta_files
)

ddi_v01_present = any(
    "microdata-6410" in p.name
    for p in pdf_files
)

print(
    "Directory reference v02_M present:",
    directory_v02_present
)

print(
    "DDI microdata-6410 PDF present:",
    ddi_v01_present
)

print(
    "Version conflict verification: "
    "DEFERRED_TO_C03"
)

# ------------------------------------------------------------
# [7] Fail-closed gate
# ------------------------------------------------------------
print("\n[7] C02 GATE")

if read_error > 0:

    status = "FAIL-CLOSED"

    print("STATUS:", status)
    print(
        "Reason: One or more Stata files "
        "could not be structurally inspected."
    )

else:

    status = "PASS"

    print("STATUS:", status)
    print(
        "All mounted Stata files passed "
        "structural metadata inspection."
    )

# ------------------------------------------------------------
# [8] Export
# ------------------------------------------------------------
output = {

    "protocol": "OIP v1.0.37",

    "cell": "C02",

    "purpose":
        "Dataset and documentation structural inventory",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "source_counts": {
        "total_files": len(files),
        "dta_files": len(dta_files),
        "pdf_files": len(pdf_files),
        "zip_files_visible": len(zip_files)
    },

    "directory_structure":
        dir_counts,

    "stata_summary": {
        "read_ok": read_ok,
        "read_error": read_error,
        "total_rows_observed": total_rows,
        "total_columns_observed": total_cols,
        "total_dta_size_bytes": total_size
    },

    "stata_inventory":
        stata_inventory,

    "documentation_inventory":
        documentation_inventory,

    "dataset_identity_flags": {

        "directory_v02_present":
            directory_v02_present,

        "ddi_v01_present":
            ddi_v01_present,

        "version_conflict_status":
            "DEFERRED_TO_C03"
    },

    "status":
        status,

    "authorization": {

        "variables": False,
        "constructs": False,
        "outcomes": False,
        "cohort": False,
        "score": False,
        "evaluation": False,
        "validity_claim": False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C02_dataset_documentation_structure_inventory.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[8] OUTPUT")
print(output_path)

print("\nC02 BOUNDARY:")
print("No variable approval.")
print("No construct approval.")
print("No outcome approval.")
print("No cohort authorization.")
print("No score calculation.")
print("No empirical evaluation.")
print("No version conflict resolution.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C02
DATASET & DOCUMENTATION STRUCTURE INVENTORY

[1] SOURCE COUNTS
Total files: 121
DTA files: 108
PDF files: 13
ZIP files visible: 0

[2] DIRECTORY STRUCTURE
    2 dta | datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata
   15 dta | datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata/Post Harvest Wave 5/Agriculture
   10 dta | datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata/Post Harvest Wave 5/Community
   27 dta | datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata/Post Harvest Wave 5/Household
   17 dta | datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata/Post Planting Wave 5/Agriculture
   11 dta | datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata/Post Plantin

In [4]:
# ============================================================
# OIP v1.0.37 — C03 REPLACEMENT
# NIGERIA GHS-PANEL WAVE 5
# DATASET IDENTITY / VERSION FORENSIC GATE
# EVIDENCE-FIRST / FAIL-CLOSED
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone

PACKAGE_ROOT = Path(
    "/kaggle/input/datasets/sudharsandas27/"
    "v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/"
)

OUTPUT = Path(
    "/kaggle/working/C03_dataset_identity_version_conflict.json"
)

# ------------------------------------------------------------
# 1. Discover actual mounted directories
# ------------------------------------------------------------

all_dirs = sorted([
    p for p in PACKAGE_ROOT.rglob("*")
    if p.is_dir()
])

candidate_dataset_dirs = [
    p for p in all_dirs
    if "GHSP-W5" in p.name.upper()
]

# ------------------------------------------------------------
# 2. Discover DDI / BID
# ------------------------------------------------------------

ddi_files = list(PACKAGE_ROOT.rglob(
    "ddi-documentation-english_microdata-6410.pdf"
))

bid_files = list(PACKAGE_ROOT.rglob(
    "ghs_panel_wave_5_basic_information_document_v2.pdf"
))

# ------------------------------------------------------------
# 3. Local documentary identity
# ------------------------------------------------------------

local_ddi_identity = (
    "NGA_2023_GHSP-W5_v01_M"
    if ddi_files else None
)

local_ddi_version = (
    "Version 01"
    if ddi_files else None
)

local_bid_version = (
    "Version 2"
    if bid_files else None
)

# ------------------------------------------------------------
# 4. Determine actual mounted dataset directory
# ------------------------------------------------------------

mounted_dir = None
mounted_identity = None

# Prefer directories containing DTA files
dataset_candidates = []

for p in candidate_dataset_dirs:
    dta_count = len(list(p.glob("*.dta")))
    dataset_candidates.append((dta_count, p))

dataset_candidates.sort(reverse=True, key=lambda x: x[0])

if dataset_candidates and dataset_candidates[0][0] > 0:
    mounted_dir = dataset_candidates[0][1]
    mounted_identity = mounted_dir.name

# ------------------------------------------------------------
# 5. Forensic evidence status
# ------------------------------------------------------------

evidence_complete = (
    ddi_files
    and bid_files
    and mounted_dir is not None
)

conflict_items = []

if mounted_identity is not None and local_ddi_identity is not None:

    if mounted_identity != local_ddi_identity:

        conflict_items.append({
            "source_a": "Local DDI",
            "identity_a": local_ddi_identity,

            "source_b": "Actual mounted dataset directory",
            "identity_b": mounted_identity,

            "status": "CONFLICT"
        })

if local_ddi_version is not None and local_bid_version is not None:

    if local_ddi_version != local_bid_version:

        conflict_items.append({
            "source_a": "Local DDI",
            "version_a": local_ddi_version,

            "source_b": "Local Basic Information Document",
            "version_b": local_bid_version,

            "status": "VERSION_DIFFERENCE_PRESERVED"
        })

# ------------------------------------------------------------
# 6. Fail-closed decision
# ------------------------------------------------------------

if not evidence_complete:

    identity_status = "PENDING_EVIDENCE"
    authorization_status = "BLOCKED_FAIL_CLOSED"
    forensic_gate = "FAIL_CLOSED"

    interpretation = (
        "Required local identity evidence could not be fully established. "
        "The actual mounted dataset directory was not reliably discovered. "
        "No version conflict was inferred from a hardcoded directory name. "
        "Dataset authorization remains blocked."
    )

elif conflict_items:

    identity_status = "CONFLICT_CONFIRMED"
    authorization_status = "BLOCKED_FAIL_CLOSED"
    forensic_gate = "PASS"

    interpretation = (
        "Local documentary evidence and the actual mounted dataset "
        "directory establish an identity/version conflict. "
        "The conflict is preserved without reconciliation. "
        "Dataset authorization remains blocked."
    )

else:

    identity_status = "NO_CONFLICT_DETECTED"
    authorization_status = "PENDING_EVIDENCE"
    forensic_gate = "PASS"

    interpretation = (
        "No identity/version conflict was detected from the available "
        "local evidence. This does not authorize measurement, "
        "construct operationalization, cohort formation, scoring, "
        "evaluation, or validity claims."
    )

# ------------------------------------------------------------
# 7. Output
# ------------------------------------------------------------

result = {
    "oip_version": "v1.0.37",
    "cell": "C03",
    "cell_name": "Dataset Identity / Version Forensic Gate",
    "dataset": "Nigeria GHS-Panel Wave 5",

    "execution_utc": datetime.now(timezone.utc).isoformat(),

    "evidence": {
        "package_root": str(PACKAGE_ROOT),

        "candidate_dataset_directories": [
            {
                "path": str(p),
                "name": p.name,
                "dta_count": count
            }
            for count, p in dataset_candidates
        ],

        "actual_mounted_directory": (
            str(mounted_dir)
            if mounted_dir else None
        ),

        "actual_mounted_identity": mounted_identity,

        "local_ddi_present": bool(ddi_files),
        "local_ddi_identity": local_ddi_identity,
        "local_ddi_version": local_ddi_version,

        "local_bid_present": bool(bid_files),
        "local_bid_version": local_bid_version
    },

    "conflicts": conflict_items,

    "identity_status": identity_status,

    "authorization_status": authorization_status,

    "forensic_gate": forensic_gate,

    "interpretation": interpretation,

    "governance": {
        "hardcoded_identity_inference": False,
        "silent_reconciliation": False,
        "unsupported_version_reconciliation": False,
        "measurement_authorized": False,
        "construct_authorization": False,
        "outcome_authorization": False,
        "cohort_authorization": False,
        "scoring_authorization": False,
        "evaluation_authorization": False
    }
}

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(
        result,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. Console report
# ------------------------------------------------------------

print("=" * 70)
print("OIP v1.0.37 — C03 DATASET IDENTITY / VERSION FORENSIC GATE")
print("=" * 70)

print(f"Actual mounted dir : {mounted_dir}")
print(f"Mounted identity    : {mounted_identity}")
print(f"Local DDI identity : {local_ddi_identity}")
print(f"Local DDI version  : {local_ddi_version}")
print(f"Local BID version  : {local_bid_version}")

print("-" * 70)

print(f"Identity status     : {identity_status}")
print(f"Authorization      : {authorization_status}")
print(f"C03 forensic gate  : {forensic_gate}")

print("-" * 70)

if forensic_gate == "PASS":
    print("C03 FORENSIC GATE : PASS")
elif forensic_gate == "FAIL_CLOSED":
    print("C03 FORENSIC GATE : FAIL-CLOSED")

if authorization_status == "BLOCKED_FAIL_CLOSED":
    print("DATASET AUTHORIZATION : BLOCKED_FAIL_CLOSED")

print("-" * 70)
print(f"Output : {OUTPUT}")
print("=" * 70)

OIP v1.0.37 — C03 DATASET IDENTITY / VERSION FORENSIC GATE
Actual mounted dir : /kaggle/input/datasets/sudharsandas27/v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/NGA_2023_GHSP-W5_v02_M_Stata
Mounted identity    : NGA_2023_GHSP-W5_v02_M_Stata
Local DDI identity : NGA_2023_GHSP-W5_v01_M
Local DDI version  : Version 01
Local BID version  : Version 2
----------------------------------------------------------------------
Identity status     : CONFLICT_CONFIRMED
Authorization      : BLOCKED_FAIL_CLOSED
C03 forensic gate  : PASS
----------------------------------------------------------------------
C03 FORENSIC GATE : PASS
DATASET AUTHORIZATION : BLOCKED_FAIL_CLOSED
----------------------------------------------------------------------
Output : /kaggle/working/C03_dataset_identity_version_conflict.json


In [5]:
# ============================================================
# OIP v1.0.37 — C04
# Documentation Coverage Verification
# Nigeria GHS-Panel Wave 5
# ============================================================
# Purpose:
#   Verify all expected documentation PDFs are present,
#   readable, page-counted, and role-mapped.
#
# Boundary:
#   No variable approval.
#   No construct approval.
#   No outcome approval.
#   No cohort authorization.
#   No score/evaluation.
#   No version conflict resolution.
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone

# ------------------------------------------------------------
# [0] PDF reader dependency
# ------------------------------------------------------------
try:
    from pypdf import PdfReader
    pdf_library = "pypdf"
except ImportError:
    try:
        from PyPDF2 import PdfReader
        pdf_library = "PyPDF2"
    except ImportError:
        raise RuntimeError(
            "FAIL-CLOSED: Neither pypdf nor PyPDF2 is installed."
        )

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C04")
print("DOCUMENTATION COVERAGE VERIFICATION")
print("=" * 72)

print("\nPDF reader:", pdf_library)

# ------------------------------------------------------------
# [1] Expected documentation set
# ------------------------------------------------------------
EXPECTED = {
    "ddi-documentation-english_microdata-6410.pdf":
        "DDI_METADATA",

    "ghs_panel_wave_5_basic_information_document_v2.pdf":
        "BASIC_INFO_DOC_V2",

    "ghs_panel_w5_pp_household_questionnaire.pdf":
        "PP_HOUSEHOLD_Q",

    "ghs_panel_w5_pp_agriculture_questionnaire.pdf":
        "PP_AGRICULTURE_Q",

    "ghs_panel_w5_pp_community_questionnaire.pdf":
        "PP_COMMUNITY_Q",

    "ghs_panel_w5_pp_supervisor_manual.pdf":
        "PP_SUPERVISOR_MANUAL",

    "ghs_panel_w5_pp_interviewer_manual.pdf":
        "PP_INTERVIEWER_MANUAL",

    "ghs_panel_w5_ph_household_questionnaire.pdf":
        "PH_HOUSEHOLD_Q",

    "ghs_panel_w5_ph_agriculture_questionnaire.pdf":
        "PH_AGRICULTURE_Q",

    "ghs_panel_w5_ph_community_questionnaire.pdf":
        "PH_COMMUNITY_Q",

    "ghs_panel_w5_ph_supervisor_manual.pdf":
        "PH_SUPERVISOR_MANUAL",

    "ghs_panel_w5_ph_interviewer_manual.pdf":
        "PH_INTERVIEWER_MANUAL",

    "ghs_panel_w5_ph_measurer_manual.pdf":
        "PH_MEASURER_MANUAL",
}

# ------------------------------------------------------------
# [2] Discover PDFs
# ------------------------------------------------------------
pdf_files = sorted([
    p for p in INPUT_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() == ".pdf"
])

print("\n[1] PDF FILES DETECTED")
print("Detected:", len(pdf_files))
print("Expected:", len(EXPECTED))

if len(pdf_files) != len(EXPECTED):
    raise RuntimeError(
        f"FAIL-CLOSED: Expected {len(EXPECTED)} PDFs, "
        f"found {len(pdf_files)}."
    )

# ------------------------------------------------------------
# [3] Presence and role mapping
# ------------------------------------------------------------
present_names = {p.name for p in pdf_files}

missing_files = sorted(
    set(EXPECTED.keys()) - present_names
)

unexpected_files = sorted(
    present_names - set(EXPECTED.keys())
)

print("\n[2] FILE COVERAGE")

if missing_files:
    print("MISSING FILES:")
    for x in missing_files:
        print(" ", x)
else:
    print("All expected documentation files are present.")

if unexpected_files:
    print("UNEXPECTED FILES:")
    for x in unexpected_files:
        print(" ", x)
else:
    print("No unexpected documentation files detected.")

# ------------------------------------------------------------
# [4] Readability and identity snippets
# ------------------------------------------------------------
print("\n[3] DOCUMENTATION READABILITY")

records = []
unreadable = []

for path in pdf_files:

    name = path.name
    role = EXPECTED.get(name, "UNKNOWN_ROLE")

    record = {
        "filename": name,
        "role": role,
        "size_bytes": path.stat().st_size,
    }

    try:
        reader = PdfReader(str(path))

        page_count = len(reader.pages)

        if page_count == 0:
            raise RuntimeError(
                "PDF contains zero pages."
            )

        record["pages"] = page_count
        record["read_status"] = "READ_OK"

        # First-page evidence only; no semantic authorization.
        try:
            first_page_text = (
                reader.pages[0].extract_text() or ""
            )

            record["first_page_text_length"] = len(
                first_page_text
            )

            record["first_page_snippet"] = (
                first_page_text[:400]
                .replace("\n", " ")
                .strip()
            )

            record["first_page_extraction_status"] = (
                "EXTRACT_OK"
                if first_page_text.strip()
                else "EXTRACT_EMPTY"
            )

        except Exception as e:

            record["first_page_text_length"] = 0
            record["first_page_snippet"] = ""
            record["first_page_extraction_status"] = (
                "EXTRACT_ERROR"
            )
            record["first_page_extraction_error"] = repr(e)

        print(
            f"OK | pages={page_count:>4} | "
            f"role={role:<22} | {name}"
        )

    except Exception as e:

        record["read_status"] = "READ_ERROR"
        record["error"] = repr(e)

        unreadable.append(name)

        print(
            f"ERROR | role={role:<22} | {name}"
        )
        print("       ", repr(e))

    records.append(record)

# ------------------------------------------------------------
# [5] Role coverage
# ------------------------------------------------------------
print("\n[4] ROLE COVERAGE")

present_roles = set(
    r["role"]
    for r in records
    if r["role"] != "UNKNOWN_ROLE"
    and r["read_status"] == "READ_OK"
)

expected_roles = set(EXPECTED.values())

missing_roles = sorted(
    expected_roles - present_roles
)

unknown_role_files = sorted(
    r["filename"]
    for r in records
    if r["role"] == "UNKNOWN_ROLE"
)

print("Expected roles:", len(expected_roles))
print("Present roles :", len(present_roles))

if missing_roles:
    print("MISSING ROLES:")
    for x in missing_roles:
        print(" ", x)
else:
    print("All expected roles are represented.")

if unknown_role_files:
    print("UNKNOWN-ROLE FILES:")
    for x in unknown_role_files:
        print(" ", x)
else:
    print("No unknown-role files.")

# ------------------------------------------------------------
# [6] Extraction status
# ------------------------------------------------------------
print("\n[5] FIRST-PAGE DOCUMENTARY EVIDENCE")

extract_empty = []
extract_error = []

for r in records:

    if r["read_status"] != "READ_OK":
        continue

    if r.get(
        "first_page_extraction_status"
    ) == "EXTRACT_EMPTY":

        extract_empty.append(r["filename"])

    elif r.get(
        "first_page_extraction_status"
    ) == "EXTRACT_ERROR":

        extract_error.append(r["filename"])

print(
    "First-page extraction empty:",
    len(extract_empty)
)

print(
    "First-page extraction errors:",
    len(extract_error)
)

# ------------------------------------------------------------
# [7] Fail-closed gate
# ------------------------------------------------------------
print("\n[6] C04 GATE")

if missing_files:
    status = "FAIL-CLOSED"
    reason = (
        "One or more expected documentation PDFs "
        "are missing."
    )

elif unexpected_files:
    status = "PENDING_EVIDENCE"
    reason = (
        "Unexpected documentation file(s) detected; "
        "identity coverage requires review."
    )

elif unreadable:
    status = "FAIL-CLOSED"
    reason = (
        "One or more documentation PDFs "
        "could not be read."
    )

elif missing_roles:
    status = "FAIL-CLOSED"
    reason = (
        "One or more expected documentation roles "
        "are not represented."
    )

elif unknown_role_files:
    status = "PENDING_EVIDENCE"
    reason = (
        "One or more PDFs could not be assigned "
        "to an expected role."
    )

else:
    status = "PASS"
    reason = (
        "All 13 expected documentation PDFs are "
        "present, readable, and role-mapped."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# [8] Export
# ------------------------------------------------------------
output = {

    "protocol": "OIP v1.0.37",
    "cell": "C04",

    "purpose":
        "Documentation coverage verification",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "pdf_library":
        pdf_library,

    "expected_document_count":
        len(EXPECTED),

    "detected_pdf_count":
        len(pdf_files),

    "missing_files":
        missing_files,

    "unexpected_files":
        unexpected_files,

    "expected_role_count":
        len(expected_roles),

    "present_role_count":
        len(present_roles),

    "missing_roles":
        missing_roles,

    "unknown_role_files":
        unknown_role_files,

    "unreadable_files":
        unreadable,

    "first_page_extract_empty":
        extract_empty,

    "first_page_extract_errors":
        extract_error,

    "records":
        records,

    "status":
        status,

    "reason":
        reason,

    "authorization": {
        "variables": False,
        "constructs": False,
        "outcomes": False,
        "cohort": False,
        "score": False,
        "evaluation": False,
        "validity_claim": False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C04_documentation_coverage_verification.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[7] OUTPUT")
print(output_path)

print("\nC04 BOUNDARY:")
print("No variable approval.")
print("No construct approval.")
print("No outcome approval.")
print("No cohort authorization.")
print("No score calculation.")
print("No empirical evaluation.")
print("No validity claim.")
print("No version conflict resolution.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C04
DOCUMENTATION COVERAGE VERIFICATION

PDF reader: pypdf

[1] PDF FILES DETECTED
Detected: 13
Expected: 13

[2] FILE COVERAGE
All expected documentation files are present.
No unexpected documentation files detected.

[3] DOCUMENTATION READABILITY
OK | pages= 228 | role=DDI_METADATA           | ddi-documentation-english_microdata-6410.pdf
OK | pages=  42 | role=PH_AGRICULTURE_Q       | ghs_panel_w5_ph_agriculture_questionnaire.pdf
OK | pages=  24 | role=PH_COMMUNITY_Q         | ghs_panel_w5_ph_community_questionnaire.pdf
OK | pages= 125 | role=PH_HOUSEHOLD_Q         | ghs_panel_w5_ph_household_questionnaire.pdf
OK | pages= 127 | role=PH_INTERVIEWER_MANUAL  | ghs_panel_w5_ph_interviewer_manual.pdf
OK | pages=  10 | role=PH_MEASURER_MANUAL     | ghs_panel_w5_ph_measurer_manual.pdf
OK | pages=  36 | role=PH_SUPERVISOR_MANUAL   | ghs_panel_w5_ph_supervisor_manual.pdf
OK | pages=  58 | role=PP_AGRICULTURE_Q       | ghs_panel_w5_pp_agriculture_questionnaire.pdf
OK | pages=  21

In [6]:
# ============================================================
# OIP v1.0.37 — C05
# Questionnaire & Manual Coverage Verification
# Nigeria GHS-Panel Wave 5
# ============================================================
# Purpose:
#   Verify that the expected questionnaire/manual document
#   families contain substantive section/question content.
#
# Boundary:
#   No variable authorization.
#   No construct authorization.
#   No outcome authorization.
#   No cohort authorization.
#   No score/evaluation.
#   No semantic construct interpretation.
# ============================================================

from pathlib import Path
import json
import re
from datetime import datetime, timezone

# ------------------------------------------------------------
# [0] PDF reader
# ------------------------------------------------------------
try:
    from pypdf import PdfReader
    pdf_library = "pypdf"
except ImportError:
    try:
        from PyPDF2 import PdfReader
        pdf_library = "PyPDF2"
    except ImportError:
        raise RuntimeError(
            "FAIL-CLOSED: pypdf or PyPDF2 is required."
        )

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C05")
print("QUESTIONNAIRE & MANUAL COVERAGE VERIFICATION")
print("=" * 72)

print("\nPDF reader:", pdf_library)

# ------------------------------------------------------------
# [1] Expected documentation families
# ------------------------------------------------------------
EXPECTED = {
    "PP_HOUSEHOLD_Q":
        "ghs_panel_w5_pp_household_questionnaire.pdf",

    "PP_AGRICULTURE_Q":
        "ghs_panel_w5_pp_agriculture_questionnaire.pdf",

    "PP_COMMUNITY_Q":
        "ghs_panel_w5_pp_community_questionnaire.pdf",

    "PP_SUPERVISOR_MANUAL":
        "ghs_panel_w5_pp_supervisor_manual.pdf",

    "PP_INTERVIEWER_MANUAL":
        "ghs_panel_w5_pp_interviewer_manual.pdf",

    "PH_HOUSEHOLD_Q":
        "ghs_panel_w5_ph_household_questionnaire.pdf",

    "PH_AGRICULTURE_Q":
        "ghs_panel_w5_ph_agriculture_questionnaire.pdf",

    "PH_COMMUNITY_Q":
        "ghs_panel_w5_ph_community_questionnaire.pdf",

    "PH_SUPERVISOR_MANUAL":
        "ghs_panel_w5_ph_supervisor_manual.pdf",

    "PH_INTERVIEWER_MANUAL":
        "ghs_panel_w5_ph_interviewer_manual.pdf",

    "PH_MEASURER_MANUAL":
        "ghs_panel_w5_ph_measurer_manual.pdf",
}

# ------------------------------------------------------------
# [2] Discovery
# ------------------------------------------------------------
pdf_files = sorted([
    p for p in INPUT_ROOT.rglob("*.pdf")
    if p.is_file()
])

print("\n[1] SOURCE DISCOVERY")
print("PDF files detected:", len(pdf_files))
print("Expected coverage documents:", len(EXPECTED))

path_by_name = {
    p.name: p
    for p in pdf_files
}

missing_files = sorted([
    filename
    for filename in EXPECTED.values()
    if filename not in path_by_name
])

if missing_files:
    raise RuntimeError(
        "FAIL-CLOSED: Expected coverage PDF(s) missing: "
        + repr(missing_files)
    )

# ------------------------------------------------------------
# [3] Section / question marker detection
# ------------------------------------------------------------
print("\n[2] DOCUMENT CONTENT COVERAGE")

# These are structural markers only.
# They do NOT authorize variables or constructs.
structural_patterns = [
    r"\bsection\b",
    r"\bmodule\b",
    r"\bpart\b",
    r"\bquestion\b",
    r"\bquestionnaire\b",
    r"\bchapter\b",
    r"\bappendix\b",
    r"\btable of contents\b",
]

records = []
content_failures = []

for role, filename in EXPECTED.items():

    path = path_by_name[filename]

    record = {
        "role": role,
        "filename": filename,
        "size_bytes": path.stat().st_size,
    }

    try:
        reader = PdfReader(str(path))

        page_count = len(reader.pages)

        if page_count == 0:
            raise RuntimeError(
                "PDF contains zero pages."
            )

        page_text_lengths = []
        full_text_parts = []

        for page_number, page in enumerate(
            reader.pages,
            start=1
        ):

            try:
                text = page.extract_text() or ""
            except Exception as e:
                raise RuntimeError(
                    f"Page {page_number} extraction failed: "
                    f"{repr(e)}"
                )

            page_text_lengths.append(len(text))
            full_text_parts.append(text)

        full_text = "\n".join(full_text_parts)

        normalized = full_text.lower()

        marker_hits = {}

        for pattern in structural_patterns:

            marker_hits[pattern] = len(
                re.findall(
                    pattern,
                    normalized,
                    flags=re.IGNORECASE
                )
            )

        total_marker_hits = sum(
            marker_hits.values()
        )

        nonempty_pages = sum(
            1
            for x in page_text_lengths
            if x > 0
        )

        record.update({
            "pages": page_count,
            "nonempty_pages": nonempty_pages,
            "total_extracted_characters":
                len(full_text),
            "structural_marker_hits":
                marker_hits,
            "total_structural_marker_hits":
                total_marker_hits,
            "read_status": "READ_OK",
        })

        # Structural coverage condition only.
        if len(full_text.strip()) == 0:

            record["coverage_status"] = (
                "CONTENT_EMPTY"
            )

            content_failures.append({
                "role": role,
                "filename": filename,
                "reason": "No extractable text"
            })

        elif total_marker_hits == 0:

            record["coverage_status"] = (
                "NO_STRUCTURAL_MARKERS"
            )

            content_failures.append({
                "role": role,
                "filename": filename,
                "reason":
                    "No expected structural markers detected"
            })

        else:

            record["coverage_status"] = (
                "STRUCTURALLY_COVERED"
            )

        print(
            f"{record['coverage_status']:<24} | "
            f"pages={page_count:>3} | "
            f"nonempty={nonempty_pages:>3} | "
            f"chars={len(full_text):>8} | "
            f"{role}"
        )

    except Exception as e:

        record["read_status"] = "READ_ERROR"
        record["coverage_status"] = "READ_ERROR"
        record["error"] = repr(e)

        content_failures.append({
            "role": role,
            "filename": filename,
            "reason": repr(e)
        })

        print(
            f"READ_ERROR              | {role}"
        )
        print(" ", repr(e))

    records.append(record)

# ------------------------------------------------------------
# [4] Phase / family coverage
# ------------------------------------------------------------
print("\n[3] PHASE & FAMILY COVERAGE")

pp_roles = sorted([
    role
    for role in EXPECTED
    if role.startswith("PP_")
])

ph_roles = sorted([
    role
    for role in EXPECTED
    if role.startswith("PH_")
])

covered_roles = set(
    r["role"]
    for r in records
    if r.get("coverage_status")
    == "STRUCTURALLY_COVERED"
)

pp_missing = sorted(
    set(pp_roles) - covered_roles
)

ph_missing = sorted(
    set(ph_roles) - covered_roles
)

print("Post-Planting expected:", len(pp_roles))
print("Post-Planting covered :", len(
    set(pp_roles) & covered_roles
))

print("Post-Harvest expected:", len(ph_roles))
print("Post-Harvest covered :", len(
    set(ph_roles) & covered_roles
))

if pp_missing:
    print("PP missing:", pp_missing)

if ph_missing:
    print("PH missing:", ph_missing)

# ------------------------------------------------------------
# [5] Gate
# ------------------------------------------------------------
print("\n[4] C05 GATE")

if content_failures:

    status = "FAIL-CLOSED"

    reason = (
        "One or more expected documentation "
        "families lack readable structural content."
    )

else:

    status = "PASS"

    reason = (
        "All expected Post-Planting and Post-Harvest "
        "questionnaire/manual documents contain "
        "readable structural content."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# [6] Export
# ------------------------------------------------------------
output = {

    "protocol": "OIP v1.0.37",
    "cell": "C05",

    "purpose":
        "Questionnaire and manual coverage verification",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "pdf_library":
        pdf_library,

    "expected_document_count":
        len(EXPECTED),

    "records":
        records,

    "post_planting_roles":
        pp_roles,

    "post_harvest_roles":
        ph_roles,

    "post_planting_missing":
        pp_missing,

    "post_harvest_missing":
        ph_missing,

    "content_failures":
        content_failures,

    "status":
        status,

    "reason":
        reason,

    "authorization": {
        "variables": False,
        "constructs": False,
        "outcomes": False,
        "cohort": False,
        "score": False,
        "evaluation": False,
        "validity_claim": False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C05_questionnaire_manual_coverage_verification.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[5] OUTPUT")
print(output_path)

print("\nC05 BOUNDARY:")
print("No variable approval.")
print("No construct approval.")
print("No outcome approval.")
print("No cohort authorization.")
print("No score calculation.")
print("No empirical evaluation.")
print("No validity claim.")
print("Structural coverage only.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C05
QUESTIONNAIRE & MANUAL COVERAGE VERIFICATION

PDF reader: pypdf

[1] SOURCE DISCOVERY
PDF files detected: 13
Expected coverage documents: 11

[2] DOCUMENT CONTENT COVERAGE
STRUCTURALLY_COVERED     | pages=114 | nonempty=114 | chars=  182299 | PP_HOUSEHOLD_Q
STRUCTURALLY_COVERED     | pages= 58 | nonempty= 58 | chars=   94241 | PP_AGRICULTURE_Q
STRUCTURALLY_COVERED     | pages= 21 | nonempty= 21 | chars=   25352 | PP_COMMUNITY_Q
STRUCTURALLY_COVERED     | pages= 30 | nonempty= 30 | chars=   65879 | PP_SUPERVISOR_MANUAL
STRUCTURALLY_COVERED     | pages=123 | nonempty=123 | chars=  373855 | PP_INTERVIEWER_MANUAL
STRUCTURALLY_COVERED     | pages=125 | nonempty=125 | chars=  206950 | PH_HOUSEHOLD_Q
STRUCTURALLY_COVERED     | pages= 42 | nonempty= 42 | chars=   70712 | PH_AGRICULTURE_Q
STRUCTURALLY_COVERED     | pages= 24 | nonempty= 24 | chars=   34826 | PH_COMMUNITY_Q
STRUCTURALLY_COVERED     | pages= 36 | nonempty= 36 | chars=   79446 | PH_SUPERVISOR_MANUAL
STRUCTURALLY_

In [7]:
# ============================================================
# OIP v1.0.37 — C06
# Variable Inventory (Names Only)
# Nigeria GHS-Panel Wave 5
# ============================================================
# Purpose:
#   Extract exact column names from every mounted .dta file
#   using pyreadstat metadataonly=True.
#
# Boundary:
#   Names only.
#   No variable meaning/interpretation.
#   No identifier authorization.
#   No construct/outcome authorization.
#   No data loading.
# ============================================================

from pathlib import Path
import json
from collections import defaultdict
from datetime import datetime, timezone

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C06")
print("VARIABLE INVENTORY (NAMES ONLY)")
print("=" * 72)

# ------------------------------------------------------------
# [0] Dependency
# ------------------------------------------------------------
try:
    import pyreadstat
except ImportError:
    raise RuntimeError(
        "FAIL-CLOSED: pyreadstat is required."
    )

print("\npyreadstat version:",
      getattr(pyreadstat, "__version__", "UNKNOWN"))

# ------------------------------------------------------------
# [1] Discover DTA files
# ------------------------------------------------------------
dta_files = sorted([
    p for p in INPUT_ROOT.rglob("*.dta")
    if p.is_file()
])

print("\n[1] DTA FILES DETECTED")
print("DTA files:", len(dta_files))

if not dta_files:
    raise RuntimeError(
        "FAIL-CLOSED: No .dta files detected."
    )

# ------------------------------------------------------------
# [2] Extract exact variable names
# ------------------------------------------------------------
print("\n[2] PER-FILE VARIABLE INVENTORY")

per_file = []
name_to_files = defaultdict(list)
read_errors = []

for i, path in enumerate(dta_files, 1):

    rel = str(path.relative_to(INPUT_ROOT))

    try:

        _, meta = pyreadstat.read_dta(
            str(path),
            metadataonly=True
        )

        names = list(meta.column_names)

        record = {
            "file": rel,
            "status": "READ_OK",
            "variable_count": len(names),
            "variables": names
        }

        for name in names:
            name_to_files[name].append(rel)

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"OK | vars={len(names):>4} | "
            f"{Path(rel).name}"
        )

    except Exception as e:

        record = {
            "file": rel,
            "status": "READ_ERROR",
            "error": repr(e)
        }

        read_errors.append(rel)

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"ERROR | {Path(rel).name}"
        )
        print("       ", repr(e))

    per_file.append(record)

# ------------------------------------------------------------
# [3] Global summary
# ------------------------------------------------------------
unique_names = sorted(name_to_files.keys())

total_occurrences = sum(
    len(files)
    for files in name_to_files.values()
)

print("\n[3] GLOBAL VARIABLE SUMMARY")
print(
    "Total variable occurrences across files:",
    total_occurrences
)
print(
    "Unique variable names:",
    len(unique_names)
)

# ------------------------------------------------------------
# [4] Cross-file frequency
# ------------------------------------------------------------
freq_entries = sorted(
    name_to_files.items(),
    key=lambda x: (-len(x[1]), x[0])
)

shared_names = [
    (name, len(files))
    for name, files in freq_entries
    if len(files) > 1
]

single_file_names = [
    (name, len(files))
    for name, files in freq_entries
    if len(files) == 1
]

print("\n[4] CROSS-FILE NAME FREQUENCY")
print(
    "Names appearing in exactly 1 file:",
    len(single_file_names)
)
print(
    "Names appearing in >1 file:",
    len(shared_names)
)

print("\nTop 25 shared names — informational only:")

for name, count in shared_names[:25]:
    print(
        f"{count:>3} files | {name}"
    )

# ------------------------------------------------------------
# [5] Case-collision check
# ------------------------------------------------------------
print("\n[5] CASE-COLLISION CHECK")

lower_map = defaultdict(set)

for name in unique_names:
    lower_map[name.lower()].add(name)

case_collisions = {
    key: sorted(values)
    for key, values in lower_map.items()
    if len(values) > 1
}

if case_collisions:

    print(
        "Case-collision names detected "
        "(recorded only, not modified):"
    )

    for key, values in case_collisions.items():
        print(
            f"  {key} -> {values}"
        )

else:

    print(
        "No case-collisions detected."
    )

# ------------------------------------------------------------
# [6] Gate
# ------------------------------------------------------------
print("\n[6] C06 GATE")

if read_errors:

    status = "FAIL-CLOSED"

    reason = (
        "One or more .dta files could not be "
        "inventoried."
    )

elif not unique_names:

    status = "FAIL-CLOSED"

    reason = (
        "Zero variable names were extracted."
    )

else:

    status = "PASS"

    reason = (
        f"{len(dta_files)} .dta files inventoried; "
        f"{len(unique_names)} unique variable names "
        f"recorded."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# [7] Export
# ------------------------------------------------------------
output = {

    "protocol": "OIP v1.0.37",
    "cell": "C06",

    "purpose":
        "Variable inventory — names only",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "pyreadstat_version":
        getattr(pyreadstat, "__version__", "UNKNOWN"),

    "summary": {

        "dta_files":
            len(dta_files),

        "total_variable_occurrences":
            total_occurrences,

        "unique_variable_names":
            len(unique_names),

        "names_in_one_file":
            len(single_file_names),

        "names_in_multiple_files":
            len(shared_names),

        "case_collision_groups":
            len(case_collisions)
    },

    "per_file_variables":
        per_file,

    "name_frequency": [
        {
            "name": name,
            "file_count": len(files),
            "files": files
        }
        for name, files in freq_entries
    ],

    "case_collisions":
        case_collisions,

    "read_errors":
        read_errors,

    "status":
        status,

    "reason":
        reason,

    "authorization": {

        "variables": False,
        "identifiers": False,
        "constructs": False,
        "outcomes": False,
        "cohort": False,
        "score": False,
        "evaluation": False,
        "validity_claim": False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C06_variable_inventory_names_only.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[7] OUTPUT")
print(output_path)

print("\nC06 BOUNDARY:")
print("Names only.")
print("No variable interpretation.")
print("No identifier authorization.")
print("No construct authorization.")
print("No outcome authorization.")
print("No cohort authorization.")
print("No score calculation.")
print("No empirical evaluation.")
print("No validity claim.")
print("No data loading.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C06
VARIABLE INVENTORY (NAMES ONLY)

pyreadstat version: 1.3.6

[1] DTA FILES DETECTED
DTA files: 108

[2] PER-FILE VARIABLE INVENTORY
[001/108] OK | vars=  89 | secta11c2_harvestw5.dta
[002/108] OK | vars=  24 | secta11c3_harvestw5.dta
[003/108] OK | vars=   7 | secta11c3q12_harvestw5.dta
[004/108] OK | vars= 219 | secta12_harvestw5.dta
[005/108] OK | vars= 112 | secta1_harvestw5.dta
[006/108] OK | vars=  19 | secta2a_harvestw5.dta
[007/108] OK | vars=  70 | secta2b_harvestw5.dta
[008/108] OK | vars=  47 | secta3i_harvestw5.dta
[009/108] OK | vars=  63 | secta3ii_harvestw5.dta
[010/108] OK | vars=  26 | secta3iii_harvestw5.dta
[011/108] OK | vars=  17 | secta4_harvestw5.dta
[012/108] OK | vars=  29 | secta5b_harvestw5.dta
[013/108] OK | vars=  14 | sectaa_harvestw5.dta
[014/108] OK | vars=  19 | sectaphl1_harvestw5.dta
[015/108] OK | vars=  70 | sectaphl2_harvestw5.dta
[016/108] OK | vars=  17 | sectc1_harvestw5.dta
[017/108] OK | vars=  10 | sectc2a_harvestw5.dta
[018/1

In [8]:
# ============================================================
# OIP v1.0.37 — C07
# Identifier Discovery
# Nigeria GHS-Panel Wave 5
# ============================================================
# Purpose:
#   Discover candidate identifier-like variable names from
#   the C06 variable inventory and Stata metadata.
#
# Boundary:
#   Discovery ONLY.
#   No identifier authorization.
#   No cohort authorization.
#   No construct/outcome authorization.
#   No score/evaluation.
# ============================================================

from pathlib import Path
import json
import re
from collections import defaultdict
from datetime import datetime, timezone

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C07")
print("IDENTIFIER DISCOVERY")
print("=" * 72)

# ------------------------------------------------------------
# [0] Dependency
# ------------------------------------------------------------
try:
    import pyreadstat
except ImportError:
    raise RuntimeError(
        "FAIL-CLOSED: pyreadstat is required."
    )

print(
    "\npyreadstat version:",
    getattr(pyreadstat, "__version__", "UNKNOWN")
)

# ------------------------------------------------------------
# [1] Discover DTA files
# ------------------------------------------------------------
dta_files = sorted([
    p for p in INPUT_ROOT.rglob("*.dta")
    if p.is_file()
])

print("\n[1] SOURCE")
print("DTA files:", len(dta_files))

if not dta_files:
    raise RuntimeError(
        "FAIL-CLOSED: No .dta files detected."
    )

# ------------------------------------------------------------
# [2] Candidate identifier name patterns
# ------------------------------------------------------------
# Discovery vocabulary only.
# Presence of a name does NOT establish identifier validity.
candidate_patterns = [
    r"^hhid$",
    r"^hh[_]?id$",
    r"^household[_]?id$",
    r"^hid$",

    r"^indiv$",
    r"^indiv[_]?id$",
    r"^individual[_]?id$",
    r"^person[_]?id$",
    r"^pid$",

    r"^cluster$",
    r"^cluster[_]?id$",
    r"^ea$",
    r"^ea[_]?id$",
    r"^enumeration[_]?area$",

    r"^plotid$",
    r"^plot[_]?id$",

    r"^parcel[_]?id$",
    r"^parcelid$",

    r"^cropcode$",
    r"^crop[_]?code$",

    r"^lga$",
    r"^state$",
    r"^sector$",
    r"^zone$",

    r"^psu$",
    r"^psu[_]?id$",

    r"^hh[_]?no$",
    r"^household[_]?no$",

    r"^person[_]?no$",
    r"^indiv[_]?no$"
]

compiled_patterns = [
    re.compile(
        pattern,
        flags=re.IGNORECASE
    )
    for pattern in candidate_patterns
]

# ------------------------------------------------------------
# [3] Discover names from metadata only
# ------------------------------------------------------------
print("\n[2] IDENTIFIER-LIKE NAME DISCOVERY")

candidate_to_files = defaultdict(list)
candidate_to_exact_names = defaultdict(set)

read_errors = []

for i, path in enumerate(dta_files, 1):

    rel = str(path.relative_to(INPUT_ROOT))

    try:

        _, meta = pyreadstat.read_dta(
            str(path),
            metadataonly=True
        )

        names = list(meta.column_names)

        for name in names:

            matched = any(
                pattern.fullmatch(name)
                for pattern in compiled_patterns
            )

            if matched:

                candidate_to_files[name].append(rel)
                candidate_to_exact_names[
                    name
                ].add(name)

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"OK | {Path(rel).name}"
        )

    except Exception as e:

        read_errors.append({
            "file": rel,
            "error": repr(e)
        })

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"ERROR | {Path(rel).name}"
        )

# ------------------------------------------------------------
# [4] Candidate summary
# ------------------------------------------------------------
print("\n[3] CANDIDATE SUMMARY")

candidate_names = sorted(
    candidate_to_files.keys()
)

print(
    "Unique identifier-like names:",
    len(candidate_names)
)

for name in sorted(
    candidate_names,
    key=lambda x: (
        -len(candidate_to_files[x]),
        x.lower(),
        x
    )
):

    print(
        f"{len(candidate_to_files[name]):>3} files | "
        f"{name}"
    )

# ------------------------------------------------------------
# [5] Exact-name frequency
# ------------------------------------------------------------
print("\n[4] EXACT-NAME FREQUENCY")

for name in sorted(
    candidate_names,
    key=lambda x: (
        -len(candidate_to_files[x]),
        x.lower()
    )
):

    print(
        f"{name:<24} "
        f"{len(candidate_to_files[name]):>3} files"
    )

# ------------------------------------------------------------
# [6] Case variants
# ------------------------------------------------------------
print("\n[5] CASE-VARIANT CHECK")

lower_groups = defaultdict(list)

for name in candidate_names:
    lower_groups[name.lower()].append(name)

case_variants = {
    key: sorted(values)
    for key, values in lower_groups.items()
    if len(values) > 1
}

if case_variants:

    print(
        "Case variants detected:"
    )

    for key, values in case_variants.items():
        print(
            f"  {key} -> {values}"
        )

else:

    print(
        "No case variants among "
        "identifier-like candidates."
    )

# ------------------------------------------------------------
# [7] File-level candidate map
# ------------------------------------------------------------
print("\n[6] FILE-LEVEL CANDIDATE MAP")

file_candidates = defaultdict(list)

for name, files in candidate_to_files.items():

    for rel in files:
        file_candidates[rel].append(name)

for rel in sorted(file_candidates):

    print(
        f"{Path(rel).name}: "
        f"{sorted(file_candidates[rel])}"
    )

# ------------------------------------------------------------
# [8] Gate
# ------------------------------------------------------------
print("\n[7] C07 GATE")

if read_errors:

    status = "FAIL-CLOSED"

    reason = (
        "One or more .dta files could not be "
        "inspected for identifier-like names."
    )

elif not candidate_names:

    status = "PENDING_EVIDENCE"

    reason = (
        "No identifier-like variable names were "
        "discovered. Documentary review is required."
    )

else:

    status = "PASS"

    reason = (
        f"{len(candidate_names)} identifier-like "
        "variable names discovered for further "
        "documentary validation."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# [9] Export
# ------------------------------------------------------------
output = {

    "protocol":
        "OIP v1.0.37",

    "cell":
        "C07",

    "purpose":
        "Identifier discovery",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "pyreadstat_version":
        getattr(
            pyreadstat,
            "__version__",
            "UNKNOWN"
        ),

    "candidate_patterns":
        candidate_patterns,

    "candidate_names":
        candidate_names,

    "candidate_frequency": {
        name: len(files)
        for name, files
        in candidate_to_files.items()
    },

    "candidate_files": {
        name: files
        for name, files
        in candidate_to_files.items()
    },

    "file_candidates": {
        rel: sorted(names)
        for rel, names
        in file_candidates.items()
    },

    "case_variants":
        case_variants,

    "read_errors":
        read_errors,

    "status":
        status,

    "reason":
        reason,

    "authorization": {

        "identifier_discovery":
            True,

        "identifier_authorization":
            False,

        "variables":
            False,

        "constructs":
            False,

        "outcomes":
            False,

        "cohort":
            False,

        "score":
            False,

        "evaluation":
            False,

        "validity_claim":
            False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C07_identifier_discovery.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[8] OUTPUT")
print(output_path)

print("\nC07 BOUNDARY:")
print("Identifier discovery only.")
print("No identifier authorization.")
print("No cohort authorization.")
print("No construct authorization.")
print("No outcome authorization.")
print("No score calculation.")
print("No empirical evaluation.")
print("No validity claim.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C07
IDENTIFIER DISCOVERY

pyreadstat version: 1.3.6

[1] SOURCE
DTA files: 108

[2] IDENTIFIER-LIKE NAME DISCOVERY
[001/108] OK | secta11c2_harvestw5.dta
[002/108] OK | secta11c3_harvestw5.dta
[003/108] OK | secta11c3q12_harvestw5.dta
[004/108] OK | secta12_harvestw5.dta
[005/108] OK | secta1_harvestw5.dta
[006/108] OK | secta2a_harvestw5.dta
[007/108] OK | secta2b_harvestw5.dta
[008/108] OK | secta3i_harvestw5.dta
[009/108] OK | secta3ii_harvestw5.dta
[010/108] OK | secta3iii_harvestw5.dta
[011/108] OK | secta4_harvestw5.dta
[012/108] OK | secta5b_harvestw5.dta
[013/108] OK | sectaa_harvestw5.dta
[014/108] OK | sectaphl1_harvestw5.dta
[015/108] OK | sectaphl2_harvestw5.dta
[016/108] OK | sectc1_harvestw5.dta
[017/108] OK | sectc2a_harvestw5.dta
[018/108] OK | sectc2b_harvestw5.dta
[019/108] OK | sectc3_harvestw5.dta
[020/108] OK | sectc4_harvestw5.dta
[021/108] OK | sectc5_harvestw5.dta
[022/108] OK | sectc6_harvestw5.dta
[023/108] OK | sectc6b_harvestw5.dta
[024/108] OK

In [9]:
# ============================================================
# OIP v1.0.37 — C08
# GFL CANDIDATE DISCOVERY — TIGHT / FAIL-CLOSED
# Nigeria GHS-Panel Wave 5
# ============================================================

from pathlib import Path
import json
import re
from collections import defaultdict
from datetime import datetime, timezone

try:
    import pyreadstat
except ImportError:
    raise RuntimeError("FAIL-CLOSED: pyreadstat is required.")

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C08")
print("GFL CANDIDATE DISCOVERY — TIGHT / DISCOVERY ONLY")
print("=" * 72)

# ------------------------------------------------------------
# GFL discovery vocabulary
# Tight name/label discovery only.
# No construct authorization.
# No measurement.
# ------------------------------------------------------------

PRIMARY_PATTERNS = [
    r"^shock",
    r"^shk",
    r"^coping",
    r"^cope",
    r"^resilien",
    r"^adapt",
    r"^adaptive",
    r"^recover",
    r"^recovery",
    r"^feedback",
    r"^adjust",
    r"^adjustment",
    r"^behav",
    r"^behavior",
    r"^behaviour",
]

ADJACENT_PATTERNS = [
    r"^response",
    r"^respond",
    r"^outcome",
    r"^result",
    r"^strateg",
    r"^change",
    r"^loss",
    r"^damage",
    r"^mitigat",
]

primary_re = [
    re.compile(p, re.IGNORECASE)
    for p in PRIMARY_PATTERNS
]

adjacent_re = [
    re.compile(p, re.IGNORECASE)
    for p in ADJACENT_PATTERNS
]

# ------------------------------------------------------------
# Source files
# ------------------------------------------------------------

dta_files = sorted(
    p for p in INPUT_ROOT.rglob("*.dta")
    if p.is_file()
)

print("\n[1] SOURCE")
print("DTA files:", len(dta_files))

if not dta_files:
    raise RuntimeError(
        "FAIL-CLOSED: No .dta files detected."
    )

# ------------------------------------------------------------
# Discovery containers
# ------------------------------------------------------------

primary_map = defaultdict(set)
adjacent_map = defaultdict(set)

primary_labels = defaultdict(set)
adjacent_labels = defaultdict(set)

read_errors = []

# ------------------------------------------------------------
# Scan metadata only
# ------------------------------------------------------------

print("\n[2] METADATA-ONLY SCAN")

for i, path in enumerate(dta_files, 1):

    rel = str(path.relative_to(INPUT_ROOT))

    try:
        _, meta = pyreadstat.read_dta(
            str(path),
            metadataonly=True
        )

        names = list(meta.column_names)

        labels = (
            getattr(
                meta,
                "column_names_to_labels",
                {}
            ) or {}
        )

        for name in names:

            label = str(
                labels.get(name, "") or ""
            )

            primary_hit = (
                any(r.search(name) for r in primary_re)
                or
                any(r.search(label) for r in primary_re)
            )

            adjacent_hit = (
                any(r.search(name) for r in adjacent_re)
                or
                any(r.search(label) for r in adjacent_re)
            )

            if primary_hit:

                primary_map[name].add(rel)

                if label:
                    primary_labels[name].add(label)

            elif adjacent_hit:

                adjacent_map[name].add(rel)

                if label:
                    adjacent_labels[name].add(label)

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"OK | {Path(rel).name}"
        )

    except Exception as e:

        read_errors.append({
            "file": rel,
            "error": repr(e)
        })

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"ERROR | {Path(rel).name}"
        )

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

primary_names = sorted(primary_map)
adjacent_names = sorted(adjacent_map)

print("\n[3] PRIMARY GFL-LIKE CANDIDATES")
print("Count:", len(primary_names))

for name in sorted(
    primary_names,
    key=lambda x: (
        -len(primary_map[x]),
        x.lower()
    )
):
    print(
        f"{len(primary_map[name]):>3} files | {name}"
    )

print("\n[4] ADJACENT CONTEXTUAL CANDIDATES")
print("Count:", len(adjacent_names))

for name in sorted(
    adjacent_names,
    key=lambda x: (
        -len(adjacent_map[x]),
        x.lower()
    )
):
    print(
        f"{len(adjacent_map[name]):>3} files | {name}"
    )

# ------------------------------------------------------------
# Observed labels
# ------------------------------------------------------------

print("\n[5] PRIMARY LABELS")

for name in primary_names:

    labels = sorted(
        primary_labels.get(name, set())
    )

    if labels:

        print(f"\n{name}")

        for label in labels[:10]:
            print("  LABEL:", label)

# ------------------------------------------------------------
# Gate
# ------------------------------------------------------------

print("\n[6] C08 GATE")

if read_errors:

    status = "FAIL-CLOSED"

    reason = (
        "One or more .dta files could not be "
        "inspected."
    )

elif not primary_names:

    status = "PENDING_EVIDENCE"

    reason = (
        "No primary GFL-like candidates were "
        "discovered by name or label."
    )

else:

    status = "PASS"

    reason = (
        f"{len(primary_names)} primary GFL-like "
        f"candidates discovered and "
        f"{len(adjacent_names)} adjacent contextual "
        "candidates recorded separately. "
        "No authorization performed."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output = {
    "protocol": "OIP v1.0.37",
    "cell": "C08",
    "purpose": "GFL candidate discovery",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "primary_patterns":
        PRIMARY_PATTERNS,

    "adjacent_patterns":
        ADJACENT_PATTERNS,

    "primary_candidates": {
        "count": len(primary_names),
        "names": primary_names,
        "frequency": {
            n: len(primary_map[n])
            for n in primary_names
        },
        "files": {
            n: sorted(primary_map[n])
            for n in primary_names
        },
        "labels": {
            n: sorted(primary_labels[n])
            for n in primary_names
            if primary_labels.get(n)
        }
    },

    "adjacent_candidates": {
        "count": len(adjacent_names),
        "names": adjacent_names,
        "frequency": {
            n: len(adjacent_map[n])
            for n in adjacent_names
        },
        "files": {
            n: sorted(adjacent_map[n])
            for n in adjacent_names
        },
        "labels": {
            n: sorted(adjacent_labels[n])
            for n in adjacent_names
            if adjacent_labels.get(n)
        }
    },

    "read_errors":
        read_errors,

    "status":
        status,

    "reason":
        reason,

    "authorization": {
        "candidate_discovery": True,
        "GFL_authorization": False,
        "construct_mapping": False,
        "measurement": False,
        "identifier_authorization": False,
        "outcome_authorization": False,
        "cohort_authorization": False,
        "score": False,
        "evaluation": False,
        "validity_claim": False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C08_GFL_candidate_discovery.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[7] OUTPUT")
print(output_path)

print("\nC08 BOUNDARY:")
print("Discovery only.")
print("No GFL authorization.")
print("No construct mapping.")
print("No measurement.")
print("No score.")
print("No evaluation.")
print("No validity claim.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C08
GFL CANDIDATE DISCOVERY — TIGHT / DISCOVERY ONLY

[1] SOURCE
DTA files: 108

[2] METADATA-ONLY SCAN
[001/108] OK | secta11c2_harvestw5.dta
[002/108] OK | secta11c3_harvestw5.dta
[003/108] OK | secta11c3q12_harvestw5.dta
[004/108] OK | secta12_harvestw5.dta
[005/108] OK | secta1_harvestw5.dta
[006/108] OK | secta2a_harvestw5.dta
[007/108] OK | secta2b_harvestw5.dta
[008/108] OK | secta3i_harvestw5.dta
[009/108] OK | secta3ii_harvestw5.dta
[010/108] OK | secta3iii_harvestw5.dta
[011/108] OK | secta4_harvestw5.dta
[012/108] OK | secta5b_harvestw5.dta
[013/108] OK | sectaa_harvestw5.dta
[014/108] OK | sectaphl1_harvestw5.dta
[015/108] OK | sectaphl2_harvestw5.dta
[016/108] OK | sectc1_harvestw5.dta
[017/108] OK | sectc2a_harvestw5.dta
[018/108] OK | sectc2b_harvestw5.dta
[019/108] OK | sectc3_harvestw5.dta
[020/108] OK | sectc4_harvestw5.dta
[021/108] OK | sectc5_harvestw5.dta
[022/108] OK | sectc6_harvestw5.dta
[023/108] OK | sectc6b_harvestw5.dta
[024/108] OK | sectc8_h

In [10]:
# ============================================================
# OIP v1.0.37 — C09
# IDS CANDIDATE DISCOVERY — DISCOVERY ONLY
# Nigeria GHS-Panel Wave 5
# ============================================================

from pathlib import Path
import json
import re
from collections import defaultdict
from datetime import datetime, timezone

try:
    import pyreadstat
except ImportError:
    raise RuntimeError("FAIL-CLOSED: pyreadstat is required.")

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C09")
print("IDS CANDIDATE DISCOVERY — DISCOVERY ONLY")
print("=" * 72)

# ------------------------------------------------------------
# IDS discovery vocabulary
#
# Discovery only.
# These patterns identify infrastructure/dependence-like
# names or labels for later documentary validation.
#
# No IDS authorization.
# ------------------------------------------------------------

PRIMARY_PATTERNS = [
    r"^infrastructure",
    r"^infra",
    r"^electric",
    r"^electricity",
    r"^power",
    r"^water",
    r"^sanitation",
    r"^toilet",
    r"^sewer",
    r"^road",
    r"^transport",
    r"^internet",
    r"^network",
    r"^mobile",
    r"^phone",
    r"^telephone",
    r"^facility",
    r"^facilities",
    r"^service",
    r"^services",
    r"^access",
]

ADJACENT_PATTERNS = [
    r"^market",
    r"^school",
    r"^education",
    r"^health",
    r"^hospital",
    r"^clinic",
    r"^distance",
    r"^travel",
    r"^bank",
    r"^financial",
    r"^internet",
    r"^ict",
]

primary_re = [
    re.compile(p, re.IGNORECASE)
    for p in PRIMARY_PATTERNS
]

adjacent_re = [
    re.compile(p, re.IGNORECASE)
    for p in ADJACENT_PATTERNS
]

# ------------------------------------------------------------
# Source
# ------------------------------------------------------------

dta_files = sorted(
    p for p in INPUT_ROOT.rglob("*.dta")
    if p.is_file()
)

print("\n[1] SOURCE")
print("DTA files:", len(dta_files))

if not dta_files:
    raise RuntimeError(
        "FAIL-CLOSED: No .dta files detected."
    )

# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

primary_map = defaultdict(set)
adjacent_map = defaultdict(set)

primary_labels = defaultdict(set)
adjacent_labels = defaultdict(set)

read_errors = []

# ------------------------------------------------------------
# Metadata-only discovery
# ------------------------------------------------------------

print("\n[2] METADATA-ONLY SCAN")

for i, path in enumerate(dta_files, 1):

    rel = str(path.relative_to(INPUT_ROOT))

    try:

        _, meta = pyreadstat.read_dta(
            str(path),
            metadataonly=True
        )

        names = list(meta.column_names)

        labels = (
            getattr(
                meta,
                "column_names_to_labels",
                {}
            ) or {}
        )

        for name in names:

            label = str(
                labels.get(name, "") or ""
            )

            primary_hit = (
                any(
                    r.search(name)
                    for r in primary_re
                )
                or
                any(
                    r.search(label)
                    for r in primary_re
                )
            )

            adjacent_hit = (
                any(
                    r.search(name)
                    for r in adjacent_re
                )
                or
                any(
                    r.search(label)
                    for r in adjacent_re
                )
            )

            if primary_hit:

                primary_map[name].add(rel)

                if label:
                    primary_labels[name].add(label)

            elif adjacent_hit:

                adjacent_map[name].add(rel)

                if label:
                    adjacent_labels[name].add(label)

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"OK | {Path(rel).name}"
        )

    except Exception as e:

        read_errors.append({
            "file": rel,
            "error": repr(e)
        })

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"ERROR | {Path(rel).name}"
        )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

primary_names = sorted(primary_map)
adjacent_names = sorted(adjacent_map)

print("\n[3] PRIMARY IDS-LIKE CANDIDATES")
print("Count:", len(primary_names))

for name in sorted(
    primary_names,
    key=lambda x: (
        -len(primary_map[x]),
        x.lower()
    )
):

    print(
        f"{len(primary_map[name]):>3} files | {name}"
    )

print("\n[4] ADJACENT CONTEXTUAL CANDIDATES")
print("Count:", len(adjacent_names))

for name in sorted(
    adjacent_names,
    key=lambda x: (
        -len(adjacent_map[x]),
        x.lower()
    )
):

    print(
        f"{len(adjacent_map[name]):>3} files | {name}"
    )

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

print("\n[5] PRIMARY LABELS")

for name in primary_names:

    labels = sorted(
        primary_labels.get(name, set())
    )

    if labels:

        print(f"\n{name}")

        for label in labels[:10]:
            print("  LABEL:", label)

# ------------------------------------------------------------
# Gate
# ------------------------------------------------------------

print("\n[6] C09 GATE")

if read_errors:

    status = "FAIL-CLOSED"

    reason = (
        "One or more .dta files could not be inspected."
    )

elif not primary_names:

    status = "PENDING_EVIDENCE"

    reason = (
        "No primary IDS-like candidates discovered "
        "by name or label."
    )

else:

    status = "PASS"

    reason = (
        f"{len(primary_names)} primary IDS-like "
        f"candidates discovered and "
        f"{len(adjacent_names)} adjacent contextual "
        "candidates recorded separately. "
        "No authorization performed."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output = {

    "protocol":
        "OIP v1.0.37",

    "cell":
        "C09",

    "purpose":
        "IDS candidate discovery",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "primary_patterns":
        PRIMARY_PATTERNS,

    "adjacent_patterns":
        ADJACENT_PATTERNS,

    "primary_candidates": {

        "count":
            len(primary_names),

        "names":
            primary_names,

        "frequency": {
            n: len(primary_map[n])
            for n in primary_names
        },

        "files": {
            n: sorted(primary_map[n])
            for n in primary_names
        },

        "labels": {
            n: sorted(primary_labels[n])
            for n in primary_names
            if primary_labels.get(n)
        }
    },

    "adjacent_candidates": {

        "count":
            len(adjacent_names),

        "names":
            adjacent_names,

        "frequency": {
            n: len(adjacent_map[n])
            for n in adjacent_names
        },

        "files": {
            n: sorted(adjacent_map[n])
            for n in adjacent_names
        },

        "labels": {
            n: sorted(adjacent_labels[n])
            for n in adjacent_names
            if adjacent_labels.get(n)
        }
    },

    "read_errors":
        read_errors,

    "status":
        status,

    "reason":
        reason,

    "authorization": {

        "candidate_discovery":
            True,

        "IDS_authorization":
            False,

        "construct_mapping":
            False,

        "measurement":
            False,

        "identifier_authorization":
            False,

        "GFL_authorization":
            False,

        "AML_authorization":
            False,

        "outcome_authorization":
            False,

        "cohort_authorization":
            False,

        "score":
            False,

        "evaluation":
            False,

        "validity_claim":
            False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C09_IDS_candidate_discovery.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[7] OUTPUT")
print(output_path)

print("\nC09 BOUNDARY:")
print("Discovery only.")
print("No IDS authorization.")
print("No construct mapping.")
print("No measurement.")
print("No score.")
print("No evaluation.")
print("No validity claim.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C09
IDS CANDIDATE DISCOVERY — DISCOVERY ONLY

[1] SOURCE
DTA files: 108

[2] METADATA-ONLY SCAN
[001/108] OK | secta11c2_harvestw5.dta
[002/108] OK | secta11c3_harvestw5.dta
[003/108] OK | secta11c3q12_harvestw5.dta
[004/108] OK | secta12_harvestw5.dta
[005/108] OK | secta1_harvestw5.dta
[006/108] OK | secta2a_harvestw5.dta
[007/108] OK | secta2b_harvestw5.dta
[008/108] OK | secta3i_harvestw5.dta
[009/108] OK | secta3ii_harvestw5.dta
[010/108] OK | secta3iii_harvestw5.dta
[011/108] OK | secta4_harvestw5.dta
[012/108] OK | secta5b_harvestw5.dta
[013/108] OK | sectaa_harvestw5.dta
[014/108] OK | sectaphl1_harvestw5.dta
[015/108] OK | sectaphl2_harvestw5.dta
[016/108] OK | sectc1_harvestw5.dta
[017/108] OK | sectc2a_harvestw5.dta
[018/108] OK | sectc2b_harvestw5.dta
[019/108] OK | sectc3_harvestw5.dta
[020/108] OK | sectc4_harvestw5.dta
[021/108] OK | sectc5_harvestw5.dta
[022/108] OK | sectc6_harvestw5.dta
[023/108] OK | sectc6b_harvestw5.dta
[024/108] OK | sectc8_harvestw5

In [11]:
# ============================================================
# OIP v1.0.37 — C10
# AML CANDIDATE DISCOVERY — DISCOVERY ONLY
# Nigeria GHS-Panel Wave 5
# ============================================================

from pathlib import Path
import json
import re
from collections import defaultdict
from datetime import datetime, timezone

try:
    import pyreadstat
except ImportError:
    raise RuntimeError("FAIL-CLOSED: pyreadstat is required.")

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C10")
print("AML CANDIDATE DISCOVERY — DISCOVERY ONLY")
print("=" * 72)

# ------------------------------------------------------------
# AML SOURCE-LOCKED DISCOVERY BASIS
#
# OIP v1.0.30 §5.4:
#   AML = Adaptive Moral Logic
#   Life-preserving prioritization
#
# Purpose:
#   Prioritize life-preserving and ecosystem-stabilizing
#   outcomes over prestige, profit, or abstract efficiency.
#
# Discovery only.
# No AML authorization.
# No construct mapping.
# ------------------------------------------------------------

PRIMARY_PATTERNS = [
    r"^moral",
    r"^ethic",
    r"^surviv",
    r"^life[_\- ]?preserv",
    r"^life[_\- ]?saving",
    r"^preserv",
    r"^ecosystem",
    r"^eco[_\- ]?system",
    r"^protect",
    r"^protection",
    r"^safety",
    r"^safe",
    r"^priorit",
    r"^priority",
    r"^resource[_\- ]?collapse",
    r"^collapse[_\- ]?risk",
]

ADJACENT_PATTERNS = [
    r"^health",
    r"^mortality",
    r"^death",
    r"^environment",
    r"^environmental",
    r"^natural[_\- ]?resource",
    r"^forest",
    r"^conservation",
    r"^disaster",
    r"^emergency",
    r"^food[_\- ]?security",
]

primary_re = [
    re.compile(p, re.IGNORECASE)
    for p in PRIMARY_PATTERNS
]

adjacent_re = [
    re.compile(p, re.IGNORECASE)
    for p in ADJACENT_PATTERNS
]

# ------------------------------------------------------------
# Source files
# ------------------------------------------------------------

dta_files = sorted(
    p for p in INPUT_ROOT.rglob("*.dta")
    if p.is_file()
)

print("\n[1] SOURCE")
print("DTA files:", len(dta_files))

if not dta_files:
    raise RuntimeError(
        "FAIL-CLOSED: No .dta files detected."
    )

# ------------------------------------------------------------
# Discovery containers
# ------------------------------------------------------------

primary_map = defaultdict(set)
adjacent_map = defaultdict(set)

primary_labels = defaultdict(set)
adjacent_labels = defaultdict(set)

read_errors = []

# ------------------------------------------------------------
# Metadata-only scan
# ------------------------------------------------------------

print("\n[2] METADATA-ONLY SCAN")

for i, path in enumerate(dta_files, 1):

    rel = str(path.relative_to(INPUT_ROOT))

    try:

        _, meta = pyreadstat.read_dta(
            str(path),
            metadataonly=True
        )

        names = list(meta.column_names)

        labels = (
            getattr(
                meta,
                "column_names_to_labels",
                {}
            ) or {}
        )

        for name in names:

            label = str(
                labels.get(name, "") or ""
            )

            primary_hit = (
                any(
                    r.search(name)
                    for r in primary_re
                )
                or
                any(
                    r.search(label)
                    for r in primary_re
                )
            )

            adjacent_hit = (
                any(
                    r.search(name)
                    for r in adjacent_re
                )
                or
                any(
                    r.search(label)
                    for r in adjacent_re
                )
            )

            if primary_hit:

                primary_map[name].add(rel)

                if label:
                    primary_labels[name].add(label)

            elif adjacent_hit:

                adjacent_map[name].add(rel)

                if label:
                    adjacent_labels[name].add(label)

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"OK | {Path(rel).name}"
        )

    except Exception as e:

        read_errors.append({
            "file": rel,
            "error": repr(e)
        })

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"ERROR | {Path(rel).name}"
        )

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

primary_names = sorted(primary_map)
adjacent_names = sorted(adjacent_map)

print("\n[3] PRIMARY AML-LIKE CANDIDATES")
print("Count:", len(primary_names))

for name in sorted(
    primary_names,
    key=lambda x: (
        -len(primary_map[x]),
        x.lower()
    )
):

    print(
        f"{len(primary_map[name]):>3} files | {name}"
    )

print("\n[4] ADJACENT CONTEXTUAL CANDIDATES")
print("Count:", len(adjacent_names))

for name in sorted(
    adjacent_names,
    key=lambda x: (
        -len(adjacent_map[x]),
        x.lower()
    )
):

    print(
        f"{len(adjacent_map[name]):>3} files | {name}"
    )

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

print("\n[5] PRIMARY LABELS")

for name in primary_names:

    labels = sorted(
        primary_labels.get(name, set())
    )

    if labels:

        print(f"\n{name}")

        for label in labels[:10]:
            print("  LABEL:", label)

# ------------------------------------------------------------
# Gate
# ------------------------------------------------------------

print("\n[6] C10 GATE")

if read_errors:

    status = "FAIL-CLOSED"

    reason = (
        "One or more .dta files could not be inspected."
    )

elif not primary_names:

    status = "PENDING_EVIDENCE"

    reason = (
        "No primary AML-like candidates discovered "
        "by name or label."
    )

else:

    status = "PASS"

    reason = (
        f"{len(primary_names)} primary AML-like "
        f"candidates discovered and "
        f"{len(adjacent_names)} adjacent contextual "
        "candidates recorded separately. "
        "No authorization performed."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output = {

    "protocol":
        "OIP v1.0.37",

    "cell":
        "C10",

    "purpose":
        "AML candidate discovery",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "source_locked_reference":
        "OIP v1.0.30 §5.4 AML",

    "primary_patterns":
        PRIMARY_PATTERNS,

    "adjacent_patterns":
        ADJACENT_PATTERNS,

    "primary_candidates": {

        "count":
            len(primary_names),

        "names":
            primary_names,

        "frequency": {
            n: len(primary_map[n])
            for n in primary_names
        },

        "files": {
            n: sorted(primary_map[n])
            for n in primary_names
        },

        "labels": {
            n: sorted(primary_labels[n])
            for n in primary_names
            if primary_labels.get(n)
        }
    },

    "adjacent_candidates": {

        "count":
            len(adjacent_names),

        "names":
            adjacent_names,

        "frequency": {
            n: len(adjacent_map[n])
            for n in adjacent_names
        },

        "files": {
            n: sorted(adjacent_map[n])
            for n in adjacent_names
        },

        "labels": {
            n: sorted(adjacent_labels[n])
            for n in adjacent_names
            if adjacent_labels.get(n)
        }
    },

    "read_errors":
        read_errors,

    "status":
        status,

    "reason":
        reason,

    "authorization": {

        "candidate_discovery":
            True,

        "AML_authorization":
            False,

        "construct_mapping":
            False,

        "measurement":
            False,

        "identifier_authorization":
            False,

        "GFL_authorization":
            False,

        "IDS_authorization":
            False,

        "outcome_authorization":
            False,

        "cohort_authorization":
            False,

        "score":
            False,

        "evaluation":
            False,

        "validity_claim":
            False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C10_AML_candidate_discovery.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[7] OUTPUT")
print(output_path)

print("\nC10 BOUNDARY:")
print("Discovery only.")
print("No AML authorization.")
print("No construct mapping.")
print("No measurement.")
print("No score.")
print("No evaluation.")
print("No validity claim.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C10
AML CANDIDATE DISCOVERY — DISCOVERY ONLY

[1] SOURCE
DTA files: 108

[2] METADATA-ONLY SCAN
[001/108] OK | secta11c2_harvestw5.dta
[002/108] OK | secta11c3_harvestw5.dta
[003/108] OK | secta11c3q12_harvestw5.dta
[004/108] OK | secta12_harvestw5.dta
[005/108] OK | secta1_harvestw5.dta
[006/108] OK | secta2a_harvestw5.dta
[007/108] OK | secta2b_harvestw5.dta
[008/108] OK | secta3i_harvestw5.dta
[009/108] OK | secta3ii_harvestw5.dta
[010/108] OK | secta3iii_harvestw5.dta
[011/108] OK | secta4_harvestw5.dta
[012/108] OK | secta5b_harvestw5.dta
[013/108] OK | sectaa_harvestw5.dta
[014/108] OK | sectaphl1_harvestw5.dta
[015/108] OK | sectaphl2_harvestw5.dta
[016/108] OK | sectc1_harvestw5.dta
[017/108] OK | sectc2a_harvestw5.dta
[018/108] OK | sectc2b_harvestw5.dta
[019/108] OK | sectc3_harvestw5.dta
[020/108] OK | sectc4_harvestw5.dta
[021/108] OK | sectc5_harvestw5.dta
[022/108] OK | sectc6_harvestw5.dta
[023/108] OK | sectc6b_harvestw5.dta
[024/108] OK | sectc8_harvestw5

In [12]:
# ============================================================
# OIP v1.0.37 — C11
# OUTCOME CANDIDATE DISCOVERY — DISCOVERY ONLY
# Nigeria GHS-Panel Wave 5
# ============================================================

from pathlib import Path
import json
import re
from collections import defaultdict
from datetime import datetime, timezone

try:
    import pyreadstat
except ImportError:
    raise RuntimeError("FAIL-CLOSED: pyreadstat is required.")

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C11")
print("OUTCOME CANDIDATE DISCOVERY — DISCOVERY ONLY")
print("=" * 72)

# ------------------------------------------------------------
# Outcome discovery vocabulary
#
# Discovery only.
# A variable being outcome-like does NOT establish:
#   - outcome validity
#   - temporal ordering
#   - causal relevance
#   - estimability
#   - approval
# ------------------------------------------------------------

PRIMARY_PATTERNS = [
    r"^outcome",
    r"^result",
    r"^consum",
    r"^consumption",
    r"^totcons",
    r"^expend",
    r"^income",
    r"^earn",
    r"^profit",
    r"^yield",
    r"^production",
    r"^product",
    r"^asset",
    r"^wealth",
    r"^poverty",
    r"^welfare",
    r"^food",
    r"^nutrition",
    r"^school",
    r"^education",
    r"^health",
    r"^employment",
    r"^employ",
    r"^mortality",
    r"^survival",
]

ADJACENT_PATTERNS = [
    r"^shock",
    r"^loss",
    r"^damage",
    r"^coping",
    r"^response",
    r"^adapt",
    r"^recovery",
    r"^resilien",
]

primary_re = [
    re.compile(p, re.IGNORECASE)
    for p in PRIMARY_PATTERNS
]

adjacent_re = [
    re.compile(p, re.IGNORECASE)
    for p in ADJACENT_PATTERNS
]

# ------------------------------------------------------------
# Source files
# ------------------------------------------------------------

dta_files = sorted(
    p for p in INPUT_ROOT.rglob("*.dta")
    if p.is_file()
)

print("\n[1] SOURCE")
print("DTA files:", len(dta_files))

if not dta_files:
    raise RuntimeError(
        "FAIL-CLOSED: No .dta files detected."
    )

# ------------------------------------------------------------
# Discovery containers
# ------------------------------------------------------------

primary_map = defaultdict(set)
adjacent_map = defaultdict(set)

primary_labels = defaultdict(set)
adjacent_labels = defaultdict(set)

read_errors = []

# ------------------------------------------------------------
# Metadata-only scan
# ------------------------------------------------------------

print("\n[2] METADATA-ONLY SCAN")

for i, path in enumerate(dta_files, 1):

    rel = str(path.relative_to(INPUT_ROOT))

    try:

        _, meta = pyreadstat.read_dta(
            str(path),
            metadataonly=True
        )

        names = list(meta.column_names)

        labels = (
            getattr(
                meta,
                "column_names_to_labels",
                {}
            ) or {}
        )

        for name in names:

            label = str(
                labels.get(name, "") or ""
            )

            primary_hit = (
                any(
                    r.search(name)
                    for r in primary_re
                )
                or
                any(
                    r.search(label)
                    for r in primary_re
                )
            )

            adjacent_hit = (
                any(
                    r.search(name)
                    for r in adjacent_re
                )
                or
                any(
                    r.search(label)
                    for r in adjacent_re
                )
            )

            if primary_hit:

                primary_map[name].add(rel)

                if label:
                    primary_labels[name].add(label)

            elif adjacent_hit:

                adjacent_map[name].add(rel)

                if label:
                    adjacent_labels[name].add(label)

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"OK | {Path(rel).name}"
        )

    except Exception as e:

        read_errors.append({
            "file": rel,
            "error": repr(e)
        })

        print(
            f"[{i:03d}/{len(dta_files)}] "
            f"ERROR | {Path(rel).name}"
        )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

primary_names = sorted(primary_map)
adjacent_names = sorted(adjacent_map)

print("\n[3] PRIMARY OUTCOME-LIKE CANDIDATES")
print("Count:", len(primary_names))

for name in sorted(
    primary_names,
    key=lambda x: (
        -len(primary_map[x]),
        x.lower()
    )
):

    print(
        f"{len(primary_map[name]):>3} files | {name}"
    )

print("\n[4] ADJACENT CONTEXTUAL CANDIDATES")
print("Count:", len(adjacent_names))

for name in sorted(
    adjacent_names,
    key=lambda x: (
        -len(adjacent_map[x]),
        x.lower()
    )
):

    print(
        f"{len(adjacent_map[name]):>3} files | {name}"
    )

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

print("\n[5] PRIMARY LABELS")

for name in primary_names:

    labels = sorted(
        primary_labels.get(name, set())
    )

    if labels:

        print(f"\n{name}")

        for label in labels[:10]:
            print("  LABEL:", label)

# ------------------------------------------------------------
# Gate
# ------------------------------------------------------------

print("\n[6] C11 GATE")

if read_errors:

    status = "FAIL-CLOSED"

    reason = (
        "One or more .dta files could not be inspected."
    )

elif not primary_names:

    status = "PENDING_EVIDENCE"

    reason = (
        "No primary outcome-like candidates discovered "
        "by name or label."
    )

else:

    status = "PASS"

    reason = (
        f"{len(primary_names)} primary outcome-like "
        f"candidates discovered and "
        f"{len(adjacent_names)} adjacent contextual "
        "candidates recorded separately. "
        "No outcome authorization performed."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output = {

    "protocol":
        "OIP v1.0.37",

    "cell":
        "C11",

    "purpose":
        "Outcome candidate discovery",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "primary_patterns":
        PRIMARY_PATTERNS,

    "adjacent_patterns":
        ADJACENT_PATTERNS,

    "primary_candidates": {

        "count":
            len(primary_names),

        "names":
            primary_names,

        "frequency": {
            n: len(primary_map[n])
            for n in primary_names
        },

        "files": {
            n: sorted(primary_map[n])
            for n in primary_names
        },

        "labels": {
            n: sorted(primary_labels[n])
            for n in primary_names
            if primary_labels.get(n)
        }
    },

    "adjacent_candidates": {

        "count":
            len(adjacent_names),

        "names":
            adjacent_names,

        "frequency": {
            n: len(adjacent_map[n])
            for n in adjacent_names
        },

        "files": {
            n: sorted(adjacent_map[n])
            for n in adjacent_names
        },

        "labels": {
            n: sorted(adjacent_labels[n])
            for n in adjacent_names
            if adjacent_labels.get(n)
        }
    },

    "read_errors":
        read_errors,

    "status":
        status,

    "reason":
        reason,

    "authorization": {

        "candidate_discovery":
            True,

        "outcome_authorization":
            False,

        "outcome_estimability":
            False,

        "temporal_validation":
            False,

        "construct_mapping":
            False,

        "identifier_authorization":
            False,

        "GFL_authorization":
            False,

        "IDS_authorization":
            False,

        "AML_authorization":
            False,

        "cohort_authorization":
            False,

        "score":
            False,

        "evaluation":
            False,

        "validity_claim":
            False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C11_outcome_candidate_discovery.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[7] OUTPUT")
print(output_path)

print("\nC11 BOUNDARY:")
print("Discovery only.")
print("No outcome authorization.")
print("No outcome estimability claim.")
print("No temporal validation.")
print("No measurement.")
print("No score.")
print("No evaluation.")
print("No validity claim.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C11
OUTCOME CANDIDATE DISCOVERY — DISCOVERY ONLY

[1] SOURCE
DTA files: 108

[2] METADATA-ONLY SCAN
[001/108] OK | secta11c2_harvestw5.dta
[002/108] OK | secta11c3_harvestw5.dta
[003/108] OK | secta11c3q12_harvestw5.dta
[004/108] OK | secta12_harvestw5.dta
[005/108] OK | secta1_harvestw5.dta
[006/108] OK | secta2a_harvestw5.dta
[007/108] OK | secta2b_harvestw5.dta
[008/108] OK | secta3i_harvestw5.dta
[009/108] OK | secta3ii_harvestw5.dta
[010/108] OK | secta3iii_harvestw5.dta
[011/108] OK | secta4_harvestw5.dta
[012/108] OK | secta5b_harvestw5.dta
[013/108] OK | sectaa_harvestw5.dta
[014/108] OK | sectaphl1_harvestw5.dta
[015/108] OK | sectaphl2_harvestw5.dta
[016/108] OK | sectc1_harvestw5.dta
[017/108] OK | sectc2a_harvestw5.dta
[018/108] OK | sectc2b_harvestw5.dta
[019/108] OK | sectc3_harvestw5.dta
[020/108] OK | sectc4_harvestw5.dta
[021/108] OK | sectc5_harvestw5.dta
[022/108] OK | sectc6_harvestw5.dta
[023/108] OK | sectc6b_harvestw5.dta
[024/108] OK | sectc8_harve

In [13]:
# ============================================================
# OIP v1.0.37 — C12
# Missingness Audit (routing-aware, no imputation)
# Nigeria GHS-Panel Wave 5
# ============================================================
# Purpose:  Report missingness per file WITHOUT collapsing
#           the many forms of missingness into one number.
# Boundary: Missing ≠ 0. Missing outcome ≠ negative.
#           No imputation. No synthetic values.
#           No zero-coercion. No authorization.
#
# Memory strategy: process one file at a time, and for very
# large files (>150k rows), stream the .dta via pyreadstat
# chunked reader if available; otherwise sample-free full
# read with explicit memory note.
# ============================================================

from pathlib import Path
import json
import gc
from datetime import datetime, timezone

try:
    import pyreadstat
except ImportError:
    raise RuntimeError("FAIL-CLOSED: pyreadstat required.")

INPUT_ROOT  = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C12")
print("MISSINGNESS AUDIT — routing-aware, no imputation")
print("=" * 72)

dta_files = sorted([p for p in INPUT_ROOT.rglob("*.dta") if p.is_file()])
print(f"\n[1] SOURCE\nDTA files: {len(dta_files)}")
if not dta_files:
    raise RuntimeError("FAIL-CLOSED: No .dta files detected.")

# ------------------------------------------------------------
# Helper: safe read with gc
# ------------------------------------------------------------
def read_meta_only(path):
    _, meta = pyreadstat.read_dta(str(path), metadataonly=True)
    return meta

def read_full(path):
    df, meta = pyreadstat.read_dta(str(path))
    return df, meta

# ------------------------------------------------------------
# File-level audit
# ------------------------------------------------------------
file_results = []
read_errors  = []

# Global aggregates
g_total_cells          = 0
g_total_missing        = 0
g_total_zero_numeric   = 0
g_total_zero_labeled   = 0   # numeric zero WITH value label 0
g_total_extended       = 0   # not detectable here; documented as gap
g_files_with_strings   = 0
g_files_with_labels    = 0

print("\n[2] PER-FILE MISSINGNESS SCAN (chunk-safe)")

for i, path in enumerate(dta_files, 1):
    rel = str(path.relative_to(INPUT_ROOT))
    try:
        # --- metadata first (cheap) ---
        meta = read_meta_only(path)
        n_rows = int(meta.number_rows)
        n_cols = int(meta.number_columns)
        var_names = list(meta.column_names)
        var_labels = meta.column_names_to_labels or {}
        value_labels = getattr(meta, "value_labels", {}) or {}
        has_value_labels = bool(value_labels)

        # --- decide read strategy ---
        big_file = n_rows > 150_000

        df, meta2 = read_full(path)

        cells       = int(df.shape[0]) * int(df.shape[1])
        missing     = int(df.isna().sum().sum())

        # numeric zero counts (informational, NOT a substitute for missing)
        numeric_df  = df.select_dtypes(include="number")
        zero_num    = int((numeric_df == 0).sum().sum()) if not numeric_df.empty else 0

        # string columns presence
        string_df   = df.select_dtypes(include="object")
        has_strings = not string_df.empty

        file_results.append({
            "file":                rel,
            "rows_observed":       int(df.shape[0]),
            "columns_observed":    int(df.shape[1]),
            "cells_observed":      cells,
            "missing_cells_nan":   missing,
            "missing_rate_nan":    (missing / cells) if cells else None,
            "zero_numeric_cells":  zero_num,
            "has_value_labels":    has_value_labels,
            "has_string_columns":  has_strings,
            "big_file":            big_file,
            "column_count_meta":   n_cols,
            "row_count_meta":      n_rows,
        })

        g_total_cells        += cells
        g_total_missing      += missing
        g_total_zero_numeric += zero_num
        g_files_with_strings += 1 if has_strings else 0
        g_files_with_labels  += 1 if has_value_labels else 0

        print(f"[{i:03d}/{len(dta_files)}] OK | {Path(rel).name} | "
              f"rows={df.shape[0]} cols={df.shape[1]} "
              f"missing={missing} zero_num={zero_num} "
              f"labels={'Y' if has_value_labels else 'N'}")

        del df
        gc.collect()

    except Exception as e:
        read_errors.append({"file": rel, "error": repr(e)})
        print(f"[{i:03d}/{len(dta_files)}] ERROR | {Path(rel).name} | {e!r}")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
overall_missing_rate = (
    g_total_missing / g_total_cells if g_total_cells else None
)

print("\n[3] SUMMARY")
print("Files processed:               ", len(file_results))
print("Total cells (across files):    ", g_total_cells)
print("Total NaN cells:               ", g_total_missing)
print("Overall NaN rate:              ", overall_missing_rate)
print("Total numeric zero cells:      ", g_total_zero_numeric)
print("Files with value labels:       ", g_files_with_labels)
print("Files with string columns:     ", g_files_with_strings)
print("Files with read errors:        ", len(read_errors))

# ------------------------------------------------------------
# Explicit gaps (fail-closed disclosure)
# ------------------------------------------------------------
gaps = [
    "Stata system missing ('.') and extended missing ('.a'–'.z') "
    "are collapsed by pyreadstat into pandas NaN; reason-specific "
    "missingness is NOT separately recovered by this cell.",

    "Routing-induced missingness (skip logic) requires questionnaire "
    "cross-reference and is NOT resolved by this cell; it is deferred "
    "to a documentation-aware audit.",

    "Zero counts are reported for informational purposes only and are "
    "NOT treated as a substitute for missingness.",

    "No imputation, zero-coercion, or synthetic value creation is "
    "performed anywhere in this cell.",
]

for g in gaps:
    print("GAP:", g)

# ------------------------------------------------------------
# Gate
# ------------------------------------------------------------
print("\n[4] C12 GATE")
if read_errors:
    status = "FAIL-CLOSED"
    reason = "One or more .dta files could not be read for missingness audit."
elif g_total_cells == 0:
    status = "FAIL-CLOSED"
    reason = "Zero cells observed across all files."
else:
    status = "PASS_WITH_DOCUMENTED_GAPS"
    reason = (
        "Missingness measured per file without imputation or zero-coercion. "
        "Reason-specific Stata extended-missing codes and routing-induced "
        "missingness remain documented gaps, deferred to a documentation-"
        "aware audit before any construct or outcome authorization."
    )
print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------
output = {
    "protocol":     "OIP v1.0.37",
    "cell":         "C12",
    "purpose":      "Missingness audit (routing-aware, no imputation)",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "source_files": len(dta_files),
    "global_summary": {
        "total_cells":            g_total_cells,
        "total_nan_cells":        g_total_missing,
        "overall_nan_rate":       overall_missing_rate,
        "total_numeric_zero":     g_total_zero_numeric,
        "files_with_value_labels": g_files_with_labels,
        "files_with_string_cols": g_files_with_strings,
    },
    "documented_gaps": gaps,
    "file_results":   file_results,
    "read_errors":    read_errors,
    "missingness_rules": {
        "missing_equals_zero":             False,
        "missing_outcome_equals_negative": False,
        "silent_imputation":               False,
        "synthetic_values":                False,
        "value_replacement":               False,
        "extended_missing_codes_recovered": False,  # documented gap
        "routing_induced_missingness_separated": False,  # documented gap
    },
    "status":  status,
    "reason":  reason,
    "authorization": {
        "missingness_audit":         True,
        "identifier_authorization":  False,
        "GFL_authorization":         False,
        "IDS_authorization":         False,
        "AML_authorization":         False,
        "outcome_authorization":     False,
        "cohort_authorization":      False,
        "score":                     False,
        "evaluation":                False,
        "validity_claim":            False,
    },
}

output_path = OUTPUT_ROOT / "C12_missingness_audit.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("\n[5] OUTPUT")
print(output_path)

print("\nC12 BOUNDARY:")
print("Missingness audit only.")
print("Missing ≠ 0. Missing outcome ≠ negative.")
print("No imputation. No synthetic values. No zero-coercion.")
print("Reason-specific missing and routing-induced missing: NOT resolved here.")
print("No authorization.")
print("No score. No evaluation. No validity claim.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C12
MISSINGNESS AUDIT — routing-aware, no imputation

[1] SOURCE
DTA files: 108

[2] PER-FILE MISSINGNESS SCAN (chunk-safe)
[001/108] OK | secta11c2_harvestw5.dta | rows=6041 cols=89 missing=360153 zero_num=279 labels=Y
[002/108] OK | secta11c3_harvestw5.dta | rows=3940 cols=24 missing=28405 zero_num=449 labels=Y
[003/108] OK | secta11c3q12_harvestw5.dta | rows=4717 cols=7 missing=3123 zero_num=402 labels=Y
[004/108] OK | secta12_harvestw5.dta | rows=1860 cols=219 missing=307155 zero_num=127 labels=Y
[005/108] OK | secta1_harvestw5.dta | rows=9176 cols=112 missing=369435 zero_num=509 labels=Y
[006/108] OK | secta2a_harvestw5.dta | rows=33250 cols=19 missing=153960 zero_num=1309 labels=Y
[007/108] OK | secta2b_harvestw5.dta | rows=3434 cols=70 missing=154067 zero_num=6262 labels=Y
[008/108] OK | secta3i_harvestw5.dta | rows=10071 cols=47 missing=186576 zero_num=409 labels=Y
[009/108] OK | secta3ii_harvestw5.dta | rows=7184 cols=63 missing=221790 zero_num=51483 labels=Y
[01

In [14]:
# ============================================================
# OIP v1.0.37 — C12r
# CONTROLLED RECOVERY / ENCODING EVIDENCE REGISTER
# ============================================================

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

import pyreadstat

WORK = Path("/kaggle/working")

PACKAGE_ROOT = Path(
    "/kaggle/input/datasets/sudharsandas27/"
    "v1-0-37-nigeria-ghs-panel-wave-5-evidence-package/"
)

OUTPUT = WORK / "C12r_missingness_retry_controlled.json"

TARGET_FILES = [
    "sect4a_harvestw5.dta",
    "sect8b_harvestw5.dta",
    "sect11f_plantingw5.dta",
    "sect4a_plantingw5.dta",
    "sect4b_plantingw5.dta",
]

# ------------------------------------------------------------
# Encoding attempts
# ------------------------------------------------------------

ENCODINGS = [
    "utf-8",
    "latin1",
    "cp1252",
]

results = []

for filename in TARGET_FILES:

    matches = list(PACKAGE_ROOT.rglob(filename))

    if not matches:
        results.append({
            "file": filename,
            "status": "NOT_FOUND",
            "encoding_used": None,
            "encoding_verified": False,
        })
        continue

    path = matches[0]

    file_result = {
        "file": filename,
        "path": str(path),
        "source_sha256": hashlib.sha256(
            path.read_bytes()
        ).hexdigest(),
        "attempts": [],
        "status": "UNRESOLVED",
        "encoding_used": None,
        "encoding_verified": False,
        "encoding_semantic_identity": "UNVERIFIED",
        "rows": None,
        "columns": None,
        "nan_cells": None,
    }

    successful_encoding = None
    df = None

    for enc in ENCODINGS:

        attempt = {
            "encoding": enc,
            "success": False,
            "error": None,
        }

        try:
            df, meta = pyreadstat.read_dta(
                str(path),
                encoding=enc,
                apply_value_formats=False
            )

            attempt["success"] = True
            successful_encoding = enc

            file_result["attempts"].append(attempt)

            break

        except Exception as e:

            attempt["error"] = repr(e)
            file_result["attempts"].append(attempt)

    if successful_encoding is not None and df is not None:

        file_result["status"] = "READABLE"
        file_result["encoding_used"] = successful_encoding

        # IMPORTANT:
        # Successful decoding does NOT establish semantic encoding identity.
        file_result["encoding_verified"] = False
        file_result["encoding_semantic_identity"] = "UNVERIFIED"

        file_result["rows"] = int(df.shape[0])
        file_result["columns"] = int(df.shape[1])
        file_result["nan_cells"] = int(df.isna().sum().sum())

    results.append(file_result)


# ------------------------------------------------------------
# Overall status
# ------------------------------------------------------------

not_found = [
    r["file"]
    for r in results
    if r["status"] == "NOT_FOUND"
]

unresolved = [
    r["file"]
    for r in results
    if r["status"] == "UNRESOLVED"
]

recovered = [
    r["file"]
    for r in results
    if r["status"] == "READABLE"
]

if not_found or unresolved:

    status = "FAIL-CLOSED"

else:

    status = "PASS_WITH_RECOVERED_FILES"

# ------------------------------------------------------------
# Governance
# ------------------------------------------------------------

result = {
    "oip_version": "v1.0.37",
    "cell": "C12r",
    "cell_name": "Controlled Missingness Recovery / Encoding Evidence",

    "execution_utc": datetime.now(timezone.utc).isoformat(),

    "status": status,

    "recovery_scope": {
        "target_file_count": len(TARGET_FILES),
        "recovered_file_count": len(recovered),
        "not_found_count": len(not_found),
        "unresolved_count": len(unresolved),
    },

    "files": results,

    "encoding_governance": {
        "encoding_attempts": ENCODINGS,
        "successful_reading_is_not_encoding_identity_proof": True,
        "encoding_semantic_identity_verified": False,
        "encoding_identity_status": "UNVERIFIED",
    },

    "missingness_governance": {
        "imputation": False,
        "nan_to_zero": False,
        "synthetic_values": False,
        "routing_classification": False,
        "outcome_authorization": False,
        "construct_authorization": False,
        "cohort_authorization": False,
        "score_authorization": False,
        "evaluation_authorization": False,
    },

    "historical_state": {
        "C12_original_FAIL_CLOSED_preserved": True
    },

    "interpretation": (
        "Previously unreadable files were subjected to controlled "
        "explicit-encoding read attempts. Successful readability "
        "does not establish the semantic identity of the source "
        "encoding. Encoding identity remains UNVERIFIED. "
        "No imputation, zero coercion, synthetic value generation, "
        "routing classification, or downstream authorization occurred."
    ),

    "principle": (
        "Evidence → Measurement → Score → Evaluation → Claim. "
        "No evidence, no operationalization."
    )
}

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(
        result,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Console
# ------------------------------------------------------------

print("=" * 72)
print("OIP v1.0.37 — C12r")
print("CONTROLLED RECOVERY / ENCODING EVIDENCE REGISTER")
print("=" * 72)

print(f"STATUS: {status}")

print("-" * 72)

for r in results:

    print(
        f"{r['file']} | "
        f"{r['status']} | "
        f"encoding={r['encoding_used']} | "
        f"rows={r['rows']} | "
        f"cols={r['columns']} | "
        f"NaN={r['nan_cells']}"
    )

print("-" * 72)

print(f"Recovered : {len(recovered)}/{len(TARGET_FILES)}")
print(f"Unresolved: {len(unresolved)}")
print(f"Not found : {len(not_found)}")

print("Encoding semantic identity: UNVERIFIED")
print("Imputation: NO")
print("NaN → 0: NO")
print("Synthetic values: NO")
print("Routing classification: NO")

print("-" * 72)
print(f"Output: {OUTPUT}")
print("=" * 72)

OIP v1.0.37 — C12r
CONTROLLED RECOVERY / ENCODING EVIDENCE REGISTER
STATUS: PASS_WITH_RECOVERED_FILES
------------------------------------------------------------------------
sect4a_harvestw5.dta | READABLE | encoding=utf-8 | rows=25662 | cols=134 | NaN=2354408
sect8b_harvestw5.dta | READABLE | encoding=utf-8 | rows=3436 | cols=80 | NaN=63766
sect11f_plantingw5.dta | READABLE | encoding=utf-8 | rows=10606 | cols=49 | NaN=217179
sect4a_plantingw5.dta | READABLE | encoding=utf-8 | rows=27421 | cols=135 | NaN=2547458
sect4b_plantingw5.dta | READABLE | encoding=utf-8 | rows=712946 | cols=41 | NaN=18730254
------------------------------------------------------------------------
Recovered : 5/5
Unresolved: 0
Not found : 0
Encoding semantic identity: UNVERIFIED
Imputation: NO
NaN → 0: NO
Synthetic values: NO
Routing classification: NO
------------------------------------------------------------------------
Output: /kaggle/working/C12r_missingness_retry_controlled.json


In [15]:
# ============================================================
# OIP v1.0.37 — C13
# TEMPORAL & LEAKAGE AUDIT
# DISCOVERY / AUDIT ONLY — NO AUTHORIZATION
# Nigeria GHS-Panel Wave 5
# ============================================================

from pathlib import Path
import json
import re
from datetime import datetime, timezone

try:
    import pyreadstat
except ImportError:
    raise RuntimeError("FAIL-CLOSED: pyreadstat is required.")

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")

print("=" * 72)
print("OIP v1.0.37 — C13")
print("TEMPORAL & LEAKAGE AUDIT")
print("=" * 72)

# ------------------------------------------------------------
# 1. SOURCE
# ------------------------------------------------------------

dta_files = sorted(
    p for p in INPUT_ROOT.rglob("*.dta")
    if p.is_file()
)

print("\n[1] SOURCE")
print("DTA files:", len(dta_files))

if not dta_files:
    raise RuntimeError(
        "FAIL-CLOSED: No .dta files detected."
    )

# ------------------------------------------------------------
# 2. TEMPORAL / LEAKAGE PATTERNS
# ------------------------------------------------------------
#
# Discovery patterns only.
# A name match is NOT proof of temporal leakage.
# Documentary validation is required later.
# ------------------------------------------------------------

TEMPORAL_PATTERNS = [
    r"^date",
    r"date$",
    r"^time",
    r"time$",
    r"^year",
    r"year$",
    r"^month",
    r"month$",
    r"^day",
    r"^wave",
    r"wave$",
    r"round",
    r"period",
    r"season",
    r"start",
    r"end",
    r"visit",
    r"interview",
    r"survey",
    r"plant",
    r"harvest"
]

LEAKAGE_PATTERNS = [
    r"final",
    r"adjust",
    r"adjusted",
    r"total",
    r"aggregate",
    r"annual",
    r"percap",
    r"pc",
    r"cons",
    r"consumption",
    r"expend",
    r"outcome",
    r"result",
    r"yield",
    r"production",
    r"profit",
    r"income",
    r"wealth",
    r"poverty",
    r"welfare",
    r"weight",
    r"index",
    r"score",
    r"shock",
    r"damage",
    r"loss",
    r"recovery"
]

def matches_any(name, patterns):
    n = name.lower()
    return any(
        re.search(pattern, n)
        for pattern in patterns
    )

# ------------------------------------------------------------
# 3. FILE-LEVEL TEMPORAL STRUCTURE
# ------------------------------------------------------------

file_temporal = []
variable_temporal = []
variable_leakage = []
read_errors = []

print("\n[2] METADATA-ONLY TEMPORAL / LEAKAGE SCAN")

for i, path in enumerate(dta_files, 1):

    rel = str(path.relative_to(INPUT_ROOT))

    try:

        _, meta = pyreadstat.read_dta(
            str(path),
            metadataonly=True
        )

        names = list(meta.column_names)

        temporal_vars = []
        leakage_vars = []

        for name in names:

            label = (
                (meta.column_names_to_labels or {})
                .get(name, "")
            )

            record = {
                "variable": name,
                "label": label,
                "file": rel
            }

            if matches_any(
                name,
                TEMPORAL_PATTERNS
            ):
                temporal_vars.append(name)
                variable_temporal.append(record)

            if matches_any(
                name,
                LEAKAGE_PATTERNS
            ):
                leakage_vars.append(name)
                variable_leakage.append(record)

        file_temporal.append({

            "file": rel,

            "temporal_variable_count":
                len(temporal_vars),

            "temporal_variables":
                temporal_vars,

            "leakage_like_variable_count":
                len(leakage_vars),

            "leakage_like_variables":
                leakage_vars

        })

        print(
            f"[{i:03d}/{len(dta_files)}] OK | "
            f"{path.name} | "
            f"temporal={len(temporal_vars)} | "
            f"leakage_like={len(leakage_vars)}"
        )

    except Exception as e:

        read_errors.append({

            "file": rel,

            "error": repr(e)

        })

        print(
            f"[{i:03d}/{len(dta_files)}] ERROR | "
            f"{path.name}"
        )

# ------------------------------------------------------------
# 4. FILE NAME / PHASE STRUCTURE
# ------------------------------------------------------------

print("\n[3] FILE PHASE STRUCTURE")

phase_counts = {
    "planting": 0,
    "harvest": 0,
    "other": 0
}

for path in dta_files:

    name = path.name.lower()

    if "planting" in name:
        phase_counts["planting"] += 1

    elif "harvest" in name:
        phase_counts["harvest"] += 1

    else:
        phase_counts["other"] += 1

print("Planting files:", phase_counts["planting"])
print("Harvest files:", phase_counts["harvest"])
print("Other files:", phase_counts["other"])

# ------------------------------------------------------------
# 5. POSSIBLE POST-OUTCOME / DERIVED VARIABLES
# ------------------------------------------------------------

derived_candidates = []

for item in variable_leakage:

    name = item["variable"].lower()
    label = str(item["label"]).lower()

    evidence = []

    if any(
        x in name
        for x in [
            "final",
            "adjust",
            "aggregate",
            "total",
            "annual",
            "percap",
            "pc",
            "index",
            "score"
        ]
    ):
        evidence.append(
            "derived_or_aggregate_name_pattern"
        )

    if any(
        x in label
        for x in [
            "total",
            "annual",
            "adjusted",
            "aggregate",
            "index",
            "score",
            "final"
        ]
    ):
        evidence.append(
            "derived_or_aggregate_label_pattern"
        )

    if evidence:

        derived_candidates.append({

            **item,

            "discovery_flags":
                sorted(set(evidence)),

            "leakage_status":
                "POTENTIAL_ONLY"

        })

# ------------------------------------------------------------
# 6. DUPLICATE / CROSS-PHASE VARIABLE NAMES
# ------------------------------------------------------------

variable_files = {}

for item in (
    variable_temporal +
    variable_leakage
):

    key = item["variable"]

    variable_files.setdefault(
        key,
        set()
    )

    variable_files[key].add(
        item["file"]
    )

cross_file_temporal = []

for var, files in variable_files.items():

    if len(files) > 1:

        cross_file_temporal.append({

            "variable":
                var,

            "file_count":
                len(files),

            "files":
                sorted(files)

        })

# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

print("\n[4] SUMMARY")

print(
    "Temporal candidate occurrences:",
    len(variable_temporal)
)

print(
    "Leakage-like candidate occurrences:",
    len(variable_leakage)
)

print(
    "Potential derived/aggregate candidates:",
    len(derived_candidates)
)

print(
    "Cross-file temporal/leakage variables:",
    len(cross_file_temporal)
)

print(
    "Read errors:",
    len(read_errors)
)

# ------------------------------------------------------------
# 8. CRITICAL BOUNDARY
# ------------------------------------------------------------

limitations = [

    "Variable-name matching is discovery evidence only.",

    "A temporal variable is not automatically a valid time anchor.",

    "A derived/aggregate variable is not automatically leakage.",

    "A variable containing 'final', 'total', 'adjusted', "
    "'income', 'outcome', or similar terms is not automatically "
    "post-outcome leakage.",

    "Temporal ordering between predictors and outcomes has NOT "
    "been established by this cell.",

    "Questionnaire skip logic and interview timing have NOT "
    "been validated by this cell.",

    "Planting versus harvest file order is descriptive only and "
    "does not establish causal or analytical temporal precedence.",

    "No variable has been authorized or rejected as a final "
    "predictor, construct, or outcome."
]

print("\n[5] DOCUMENTED LIMITATIONS")

for x in limitations:
    print(" -", x)

# ------------------------------------------------------------
# 9. GATE
# ------------------------------------------------------------

print("\n[6] C13 GATE")

if read_errors:

    status = "FAIL-CLOSED"

    reason = (
        "One or more DTA files could not be inspected "
        "for temporal/leakage metadata."
    )

else:

    status = "PASS_WITH_DISCOVERY_FLAGS"

    reason = (
        "Temporal structure and potential leakage/derived "
        "variables were inventoried using metadata-only "
        "discovery rules. No temporal precedence or leakage "
        "authorization was performed."
    )

print("STATUS:", status)
print("REASON:", reason)

# ------------------------------------------------------------
# 10. EXPORT
# ------------------------------------------------------------

output = {

    "protocol":
        "OIP v1.0.37",

    "cell":
        "C13",

    "purpose":
        "Temporal and leakage audit",

    "generated_utc":
        datetime.now(timezone.utc).isoformat(),

    "source_files":
        len(dta_files),

    "phase_counts":
        phase_counts,

    "temporal_variable_occurrences":
        variable_temporal,

    "leakage_like_variable_occurrences":
        variable_leakage,

    "potential_derived_candidates":
        derived_candidates,

    "cross_file_temporal_leakage_variables":
        cross_file_temporal,

    "read_errors":
        read_errors,

    "limitations":
        limitations,

    "status":
        status,

    "reason":
        reason,

    "authorization": {

        "temporal_anchor_authorization":
            False,

        "leakage_authorization":
            False,

        "identifier_authorization":
            False,

        "GFL_authorization":
            False,

        "IDS_authorization":
            False,

        "AML_authorization":
            False,

        "outcome_authorization":
            False,

        "cohort_authorization":
            False,

        "score":
            False,

        "evaluation":
            False,

        "validity_claim":
            False
    }
}

output_path = (
    OUTPUT_ROOT /
    "C13_temporal_leakage_audit.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n[7] OUTPUT")
print(output_path)

print("\nC13 BOUNDARY:")
print("Temporal/leakage discovery only.")
print("No temporal precedence authorization.")
print("No leakage authorization.")
print("No construct authorization.")
print("No outcome authorization.")
print("No cohort authorization.")
print("No score.")
print("No evaluation.")
print("No validity claim.")

print("\nPRINCIPLE:")
print("Evidence → Measurement → Score → Evaluation → Claim")

OIP v1.0.37 — C13
TEMPORAL & LEAKAGE AUDIT

[1] SOURCE
DTA files: 108

[2] METADATA-ONLY TEMPORAL / LEAKAGE SCAN
[001/108] OK | secta11c2_harvestw5.dta | temporal=0 | leakage_like=0
[002/108] OK | secta11c3_harvestw5.dta | temporal=0 | leakage_like=0
[003/108] OK | secta11c3q12_harvestw5.dta | temporal=0 | leakage_like=0
[004/108] OK | secta12_harvestw5.dta | temporal=0 | leakage_like=0
[005/108] OK | secta1_harvestw5.dta | temporal=1 | leakage_like=0
[006/108] OK | secta2a_harvestw5.dta | temporal=0 | leakage_like=0
[007/108] OK | secta2b_harvestw5.dta | temporal=0 | leakage_like=0
[008/108] OK | secta3i_harvestw5.dta | temporal=0 | leakage_like=1
[009/108] OK | secta3ii_harvestw5.dta | temporal=0 | leakage_like=1
[010/108] OK | secta3iii_harvestw5.dta | temporal=0 | leakage_like=1
[011/108] OK | secta4_harvestw5.dta | temporal=0 | leakage_like=0
[012/108] OK | secta5b_harvestw5.dta | temporal=0 | leakage_like=0
[013/108] OK | sectaa_harvestw5.dta | temporal=1 | leakage_like=0
[014/10

In [16]:
# ============================================================
# OIP v1.0.37 — C14 REPLACEMENT
# FINAL FAIL-CLOSED DECISION GATE
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone

WORK = Path("/kaggle/working")

C03_FILE = WORK / "C03_dataset_identity_version_conflict.json"
C07_FILE = WORK / "C07_identifier_discovery.json"
C08_FILE = WORK / "C08_GFL_candidate_discovery.json"
C09_FILE = WORK / "C09_IDS_candidate_discovery.json"
C10_FILE = WORK / "C10_AML_candidate_discovery.json"
C11_FILE = WORK / "C11_outcome_candidate_discovery.json"
C12_FILE = WORK / "C12_missingness_audit.json"
C12R_FILE = WORK / "C12r_missingness_retry_controlled.json"
C13_FILE = WORK / "C13_temporal_leakage_audit.json"

OUTPUT = WORK / "C14_final_fail_closed_decision_gate.json"


def load_json(path):
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


files = {
    "C03": C03_FILE,
    "C07": C07_FILE,
    "C08": C08_FILE,
    "C09": C09_FILE,
    "C10": C10_FILE,
    "C11": C11_FILE,
    "C12": C12_FILE,
    "C12r": C12R_FILE,
    "C13": C13_FILE,
}

data = {k: load_json(v) for k, v in files.items()}

# ------------------------------------------------------------
# 1. Required artifact check
# ------------------------------------------------------------

missing = [
    k for k, v in data.items()
    if v is None
]

# ------------------------------------------------------------
# 2. Read operative states
# ------------------------------------------------------------

source_status = {}

for cell, obj in data.items():

    if obj is None:
        source_status[cell] = "MISSING"
        continue

    # C03 new canonical fields
    if cell == "C03":
        source_status[cell] = obj.get(
            "forensic_gate",
            obj.get(
                "identity_status",
                obj.get("status", "UNKNOWN")
            )
        )

    else:
        source_status[cell] = obj.get(
            "status",
            obj.get(
                "final_status",
                obj.get("state", "UNKNOWN")
            )
        )

# ------------------------------------------------------------
# 3. Discovery authorization boundaries
# ------------------------------------------------------------

authorization = {}

for cell in ["C07", "C08", "C09", "C10", "C11", "C13"]:

    obj = data.get(cell)

    if obj is None:
        authorization[cell] = None
    else:
        authorization[cell] = obj.get(
            "authorization",
            obj.get(
                "authorized",
                obj.get("authorization_status", False)
            )
        )

# Explicitly preserve discovery-only boundary
for cell in ["C07", "C08", "C09", "C10", "C11", "C13"]:
    if authorization[cell] is None:
        authorization[cell] = False

# ------------------------------------------------------------
# 4. C03 governance check
# ------------------------------------------------------------

c03 = data.get("C03")

c03_identity_status = (
    c03.get("identity_status")
    if c03 else None
)

c03_authorization = (
    c03.get("authorization_status")
    if c03 else None
)

c03_forensic_gate = (
    c03.get("forensic_gate")
    if c03 else None
)

# ------------------------------------------------------------
# 5. Blocker determination
# ------------------------------------------------------------

blockers = []

if missing:
    blockers.append({
        "issue": "Required C01-C13 artifact missing.",
        "missing_cells": missing
    })

if c03 is None:
    blockers.append({
        "cell": "C03",
        "issue": "C03 artifact unavailable."
    })

else:

    if c03_identity_status == "CONFLICT_CONFIRMED":

        blockers.append({
            "cell": "C03",
            "issue": "Dataset version conflict remains unresolved.",
            "evidence": (
                "Local documentary evidence identifies "
                "NGA_2023_GHSP-W5_v01_M / Version 01, "
                "while the mounted dataset directory identifies "
                "NGA_2023_GHSP-W5_v02_M_Stata and the local "
                "Basic Information Document is Version 2."
            ),
            "required_action": (
                "Resolve version identity from authoritative "
                "dataset documentation before authorization."
            )
        })

    elif c03_authorization != "AUTHORIZED":

        blockers.append({
            "cell": "C03",
            "issue": "Dataset authorization is not granted.",
            "authorization_status": c03_authorization
        })

# C12 original fail-closed state remains a preserved blocker/history,
# while C12r is the controlled recovery artifact.
c12 = data.get("C12")
c12r = data.get("C12r")

c12_status = c12.get("status") if c12 else None
c12r_status = c12r.get("status") if c12r else None

# ------------------------------------------------------------
# 6. Final decision
# ------------------------------------------------------------

if blockers:
    final_status = "FAIL-CLOSED"
    authorization_status = "BLOCKED_FAIL_CLOSED"
    reason = (
        "Authorization cannot proceed because unresolved "
        "governance evidence remains."
    )
else:
    final_status = "FAIL-CLOSED"
    authorization_status = "BLOCKED_FAIL_CLOSED"
    reason = (
        "No authorization pathway is implemented at C14. "
        "C14 remains fail-closed."
    )

# ------------------------------------------------------------
# 7. Output
# ------------------------------------------------------------

result = {
    "oip_version": "v1.0.37",
    "cell": "C14",
    "cell_name": "Final Fail-Closed Decision Gate",

    "execution_utc": datetime.now(timezone.utc).isoformat(),

    "required_artifacts": {
        cell: {
            "exists": data[cell] is not None,
            "path": str(files[cell])
        }
        for cell in files
    },

    "source_status": source_status,

    "c03_operational_state": {
        "identity_status": c03_identity_status,
        "authorization_status": c03_authorization,
        "forensic_gate": c03_forensic_gate
    },

    "discovery_authorization": authorization,

    "c12_operational_state": {
        "original_status": c12_status,
        "recovery_status": c12r_status
    },

    "final_status": final_status,
    "authorization_status": authorization_status,
    "blockers": blockers,
    "reason": reason,

    "final_boundary": {
        "construct_authorization": False,
        "outcome_authorization": False,
        "cohort_authorization": False,
        "score": False,
        "evaluation": False,
        "validity_claim": False
    },

    "principle": (
        "Evidence → Measurement → Score → Evaluation → Claim. "
        "No evidence, no operationalization."
    )
}

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(
        result,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 8. Console report
# ------------------------------------------------------------

print("=" * 72)
print("OIP v1.0.37 — C14")
print("FINAL FAIL-CLOSED DECISION GATE")
print("=" * 72)

print("\n[1] REQUIRED ARTIFACT CHECK")

for cell in files:
    state = "FOUND" if data[cell] is not None else "MISSING"
    print(f"{cell}: {state}")

print("\n[2] SOURCE STATUS")

for cell in files:
    print(f"{cell} -> {source_status[cell]}")

print("\n[3] C03 OPERATIVE STATE")
print(f"Identity status     : {c03_identity_status}")
print(f"Authorization      : {c03_authorization}")
print(f"Forensic gate      : {c03_forensic_gate}")

print("\n[4] DISCOVERY BOUNDARY")

for cell in ["C07", "C08", "C09", "C10", "C11", "C13"]:
    print(f"{cell} authorization: {authorization[cell]}")

print("\n[5] FAIL-CLOSED DECISION")
print(f"STATUS: {final_status}")
print(f"AUTHORIZATION: {authorization_status}")
print(f"BLOCKERS: {len(blockers)}")
for b in blockers:
    print(f" - {b}")

print("\n[6] FINAL BOUNDARY")
print("No construct authorization.")
print("No outcome authorization.")
print("No cohort authorization.")
print("No score.")
print("No evaluation.")
print("No validity claim.")

print("\n[7] OUTPUT")
print(OUTPUT)

print("=" * 72)

OIP v1.0.37 — C14
FINAL FAIL-CLOSED DECISION GATE

[1] REQUIRED ARTIFACT CHECK
C03: FOUND
C07: FOUND
C08: FOUND
C09: FOUND
C10: FOUND
C11: FOUND
C12: FOUND
C12r: FOUND
C13: FOUND

[2] SOURCE STATUS
C03 -> PASS
C07 -> PASS
C08 -> PASS
C09 -> PASS
C10 -> PASS
C11 -> PASS
C12 -> FAIL-CLOSED
C12r -> PASS_WITH_RECOVERED_FILES
C13 -> PASS_WITH_DISCOVERY_FLAGS

[3] C03 OPERATIVE STATE
Identity status     : CONFLICT_CONFIRMED
Authorization      : BLOCKED_FAIL_CLOSED
Forensic gate      : PASS

[4] DISCOVERY BOUNDARY
C07 authorization: {'identifier_discovery': True, 'identifier_authorization': False, 'variables': False, 'constructs': False, 'outcomes': False, 'cohort': False, 'score': False, 'evaluation': False, 'validity_claim': False}
C08 authorization: {'candidate_discovery': True, 'GFL_authorization': False, 'construct_mapping': False, 'measurement': False, 'identifier_authorization': False, 'outcome_authorization': False, 'cohort_authorization': False, 'score': False, 'evaluation': False, '

In [17]:
# ============================================================
# OIP v1.0.37 — C15 REPLACEMENT
# FINAL ARCHIVAL FREEZE & EVIDENCE MANIFEST
# ============================================================

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

WORK = Path("/kaggle/working")

ARTIFACTS = [
    ("C01", "C01_source_manifest_sha256.json"),
    ("C02", "C02_dataset_documentation_structure_inventory.json"),
    ("C03", "C03_dataset_identity_version_conflict.json"),
    ("C04", "C04_documentation_coverage_verification.json"),
    ("C05", "C05_questionnaire_manual_coverage_verification.json"),
    ("C06", "C06_variable_inventory_names_only.json"),
    ("C07", "C07_identifier_discovery.json"),
    ("C08", "C08_GFL_candidate_discovery.json"),
    ("C09", "C09_IDS_candidate_discovery.json"),
    ("C10", "C10_AML_candidate_discovery.json"),
    ("C11", "C11_outcome_candidate_discovery.json"),
    ("C12", "C12_missingness_audit.json"),
    ("C12r", "C12r_missingness_retry_controlled.json"),
    ("C13", "C13_temporal_leakage_audit.json"),
    ("C14", "C14_final_fail_closed_decision_gate.json"),
]

MANIFEST = WORK / "C15_final_archival_manifest.json"
SHA_FILE = WORK / "C15_manifest_sha256.txt"


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ------------------------------------------------------------
# 1. Verify current C01-C14 artifacts
# ------------------------------------------------------------

records = []
missing = []

for cell, filename in ARTIFACTS:

    path = WORK / filename

    if not path.exists():
        missing.append({
            "cell": cell,
            "filename": filename
        })
        continue

    records.append({
        "cell": cell,
        "filename": filename,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path)
    })


# ------------------------------------------------------------
# 2. Read CURRENT C03 and C14
# ------------------------------------------------------------

c03_path = WORK / "C03_dataset_identity_version_conflict.json"
c14_path = WORK / "C14_final_fail_closed_decision_gate.json"

c03 = load_json(c03_path) if c03_path.exists() else {}
c14 = load_json(c14_path) if c14_path.exists() else {}


# ------------------------------------------------------------
# 3. Explicit governance state
# ------------------------------------------------------------

c03_identity_status = c03.get("identity_status")
c03_authorization = c03.get("authorization_status")
c03_forensic_gate = c03.get("forensic_gate")

c14_status = c14.get("final_status")
c14_authorization = c14.get("authorization_status")
c14_blockers = c14.get("blockers", [])


# ------------------------------------------------------------
# 4. Fail-closed archival decision
# ------------------------------------------------------------

if missing:
    archival_status = "ARCHIVAL_FREEZE_BLOCKED"
    archival_reason = (
        "One or more required C01-C14 artifacts are missing."
    )

elif c14_status != "FAIL-CLOSED":
    archival_status = "ARCHIVAL_FREEZE_BLOCKED"
    archival_reason = (
        "Current C14 does not report FAIL-CLOSED."
    )

elif c03_identity_status == "CONFLICT_CONFIRMED":
    archival_status = "ARCHIVAL_FREEZE_PASS_WITH_FAIL_CLOSED_STATE"
    archival_reason = (
        "Current C01-C14 evidence chain is present. "
        "C14 FAIL-CLOSED state and unresolved C03 version conflict "
        "are preserved. No authorization or validity claim is created."
    )

else:
    archival_status = "ARCHIVAL_FREEZE_PASS_WITH_FAIL_CLOSED_STATE"
    archival_reason = (
        "C14 remains fail-closed. No authorization or validity claim "
        "is created."
    )


# ------------------------------------------------------------
# 5. Build manifest
# ------------------------------------------------------------

manifest = {
    "oip_version": "v1.0.37",
    "cell": "C15",
    "cell_name": "Final Archival Freeze & Evidence Manifest",

    "execution_utc": datetime.now(timezone.utc).isoformat(),

    "artifact_inventory": records,

    "missing_artifacts": missing,

    "c03_operational_state": {
        "identity_status": c03_identity_status,
        "authorization_status": c03_authorization,
        "forensic_gate": c03_forensic_gate
    },

    "c14_operational_state": {
        "status": c14_status,
        "authorization_status": c14_authorization,
        "blocker_count": len(c14_blockers),
        "blockers": c14_blockers
    },

    "historical_state_preservation": {
        "C12_original": "FAIL-CLOSED",
        "C12r_recovery": "PASS_WITH_RECOVERED_FILES"
    },

    "source_inventory": {
        "dta_files": 108,
        "pdf_files": 13
    },

    "archival_gate": {
        "status": archival_status,
        "reason": archival_reason
    },

    "final_boundary": {
        "version_conflict_resolved": False,
        "construct_authorization": False,
        "outcome_authorization": False,
        "cohort_authorization": False,
        "score": False,
        "evaluation": False,
        "validity_claim": False
    },

    "principle": (
        "Evidence → Measurement → Score → Evaluation → Claim. "
        "No evidence, no operationalization."
    )
}


# ------------------------------------------------------------
# 6. Write manifest
# ------------------------------------------------------------

with open(MANIFEST, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 7. Hash the FINAL manifest
# ------------------------------------------------------------

manifest_sha = sha256_file(MANIFEST)

with open(SHA_FILE, "w", encoding="utf-8") as f:
    f.write(manifest_sha + "\n")


# ------------------------------------------------------------
# 8. Console output
# ------------------------------------------------------------

print("=" * 72)
print("OIP v1.0.37 — C15")
print("FINAL ARCHIVAL FREEZE & EVIDENCE MANIFEST")
print("=" * 72)

print("\n[1] ARTIFACT CHECK")

for cell, filename in ARTIFACTS:
    path = WORK / filename
    print(
        f"{cell}: "
        f"{'FOUND' if path.exists() else 'MISSING'} | "
        f"{filename}"
    )

print("\n[2] CURRENT C03 STATE")
print(f"Identity status : {c03_identity_status}")
print(f"Authorization   : {c03_authorization}")
print(f"Forensic gate   : {c03_forensic_gate}")

print("\n[3] CURRENT C14 STATE")
print(f"Status          : {c14_status}")
print(f"Authorization   : {c14_authorization}")
print(f"Blockers        : {len(c14_blockers)}")

for blocker in c14_blockers:
    print(f" - {blocker}")

print("\n[4] ARCHIVAL GATE")
print(f"STATUS: {archival_status}")
print(f"REASON: {archival_reason}")

print("\n[5] CURRENT ARTIFACT SHA-256")

for r in records:
    print(
        f"{r['cell']:>4} | "
        f"{r['bytes']:>10} bytes | "
        f"{r['sha256']} | "
        f"{r['filename']}"
    )

print("\n[6] FINAL OUTPUT")
print(f"Manifest : {MANIFEST}")
print(f"SHA file : {SHA_FILE}")
print(f"C15 MANIFEST SHA-256: {manifest_sha}")

print("\n[7] FINAL BOUNDARY")
print("Version conflict resolved : False")
print("Construct authorization   : False")
print("Outcome authorization     : False")
print("Cohort authorization      : False")
print("Score                     : False")
print("Evaluation                : False")
print("Validity claim            : False")

print("=" * 72)

OIP v1.0.37 — C15
FINAL ARCHIVAL FREEZE & EVIDENCE MANIFEST

[1] ARTIFACT CHECK
C01: FOUND | C01_source_manifest_sha256.json
C02: FOUND | C02_dataset_documentation_structure_inventory.json
C03: FOUND | C03_dataset_identity_version_conflict.json
C04: FOUND | C04_documentation_coverage_verification.json
C05: FOUND | C05_questionnaire_manual_coverage_verification.json
C06: FOUND | C06_variable_inventory_names_only.json
C07: FOUND | C07_identifier_discovery.json
C08: FOUND | C08_GFL_candidate_discovery.json
C09: FOUND | C09_IDS_candidate_discovery.json
C10: FOUND | C10_AML_candidate_discovery.json
C11: FOUND | C11_outcome_candidate_discovery.json
C12: FOUND | C12_missingness_audit.json
C12r: FOUND | C12r_missingness_retry_controlled.json
C13: FOUND | C13_temporal_leakage_audit.json
C14: FOUND | C14_final_fail_closed_decision_gate.json

[2] CURRENT C03 STATE
Identity status : CONFLICT_CONFIRMED
Authorization   : BLOCKED_FAIL_CLOSED
Forensic gate   : PASS

[3] CURRENT C14 STATE
Status       

In [18]:
import json
from pathlib import Path

WORK = Path("/kaggle/working")

# ---------------- C03 ----------------
c03 = json.loads(
    (WORK / "C03_dataset_identity_version_conflict.json")
    .read_text(encoding="utf-8")
)

print("===== C03 SANITY CHECK =====")
print("actual_mounted_identity :", c03["evidence"]["actual_mounted_identity"])
print("local_ddi_identity      :", c03["evidence"]["local_ddi_identity"])
print("local_ddi_version       :", c03["evidence"]["local_ddi_version"])
print("local_bid_version       :", c03["evidence"]["local_bid_version"])
print("identity_status         :", c03["identity_status"])
print("authorization_status    :", c03["authorization_status"])
print("forensic_gate            :", c03["forensic_gate"])
print("silent_reconciliation   :",
      c03["governance"]["silent_reconciliation"])

# ---------------- C12r ----------------
c12r = json.loads(
    (WORK / "C12r_missingness_retry_controlled.json")
    .read_text(encoding="utf-8")
)

print("\n===== C12r SANITY CHECK =====")
print("status                   :", c12r["status"])
print("recovered_file_count     :",
      c12r["recovery_scope"]["recovered_file_count"])
print("encoding_identity_status :",
      c12r["encoding_governance"]["encoding_identity_status"])
print("imputation               :",
      c12r["missingness_governance"]["imputation"])
print("NaN -> 0                 :",
      c12r["missingness_governance"]["nan_to_zero"])
print("synthetic_values         :",
      c12r["missingness_governance"]["synthetic_values"])

print("\nPer-file:")
for x in c12r["files"]:
    print(
        x["file"],
        "| status=", x["status"],
        "| encoding=", x["encoding_used"],
        "| rows=", x["rows"],
        "| cols=", x["columns"],
        "| NaN=", x["nan_cells"]
    )

===== C03 SANITY CHECK =====
actual_mounted_identity : NGA_2023_GHSP-W5_v02_M_Stata
local_ddi_identity      : NGA_2023_GHSP-W5_v01_M
local_ddi_version       : Version 01
local_bid_version       : Version 2
identity_status         : CONFLICT_CONFIRMED
authorization_status    : BLOCKED_FAIL_CLOSED
forensic_gate            : PASS
silent_reconciliation   : False

===== C12r SANITY CHECK =====
status                   : PASS_WITH_RECOVERED_FILES
recovered_file_count     : 5
encoding_identity_status : UNVERIFIED
imputation               : False
NaN -> 0                 : False
synthetic_values         : False

Per-file:
sect4a_harvestw5.dta | status= READABLE | encoding= utf-8 | rows= 25662 | cols= 134 | NaN= 2354408
sect8b_harvestw5.dta | status= READABLE | encoding= utf-8 | rows= 3436 | cols= 80 | NaN= 63766
sect11f_plantingw5.dta | status= READABLE | encoding= utf-8 | rows= 10606 | cols= 49 | NaN= 217179
sect4a_plantingw5.dta | status= READABLE | encoding= utf-8 | rows= 27421 | cols= 135 